**Making Figures (Milestone 3) | Tyler Hilbert | April 28, 2026**

Aside from refining the tables and prepping for the PowerBI dashboard, I am creating the figures that visualize the data over time. When I started this project, I visualized three different figures that could be made with the data that I would still like to create:
- A line chart that shows the percent of each grade over time (passing, not reported, failing) - for the purpose of seeing how the grades fluctuated.
- A line chart that shows the number of needed interventions over time - to see if these have remained constant or if there have been ups and downs.
- A heat map that shows grade movement from MT grades to final grades.

Knowing what I know now about integrating plotly with PowerBI and the limitations set by the university, I need to consider how I would like to filter these figures early and how to implement them on the PowerBI dashboard. I believe that the best way to handle this would be to create individual department/subject level figures for the % line chart and heat map with filters for specific courses on each. I could then use PowerBI's button feature as a sort of filter to make buttons for each department - that way you can load the other tables while cirucmventing the "one filter" limitation. For the intervention line chart - we can build it all into the figure with a filter for the different colleges since the concern is the total number of interventions, not with the number of interventions per college.

*Loading Libraries*

In [512]:
import pandas as pd #Will be useful for any further data manipultion
import numpy as np #May be needed if we have to add additional filters/searches
import plotly.graph_objects as go #This will be the main library we use for creating the figures
import plotly.io as pio #This is needed for the importing of figures to PowerBI
import json #Needed for the conversion of figures to PowerBI

**Making the % Over Time Line Chart**

The first figure I'd like to make is converting the summary table (which shows the total percent of students passing/failing/no reported MT grades) over time. The purpose of this figure will be to look for any trends in grade data and help identify courses that may be pain points for students that should be addressed. We will first pull the data in - we will be using the *mtfullsummary.csv* file we made last week for this figure, as it will contain all of the data points.

I want to note that, due to the anonymization of the dataset, the data points will be all over the place. When we switch to real grade data, there will be more consistent trends of data points.

*Loading the Data*

In [513]:
#Loading the Data Pt. 1 (Pulling in the dataset)
mtpct = pd.read_csv("mtfullsummary.csv")
mtpct.head()

,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,THEA 28661,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0,1.0,100.00%
1,MDJ 11788,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0,1.0,100.00%
2,DAN 22742,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0,1.0,100.00%
3,THEA 14873,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0,1.0,100.00%
4,MUS 20682,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0,1.0,100.00%


In [514]:
#Loading the Data Pt. 2 (Checking the Shape)
mtpct.shape

(778, 11)

In [515]:
#Loading the Data Pt. 3 (Checking Unique Value Counts)
mtpct["Course Code"].value_counts()

Course Code
VCD 17656     8
MDJ 25865     8
EMAT 12670    8
ARCH 22860    8
FDM 17764     8
             ..
THEA 10123    1
THEA 10321    1
MUS 11618     1
THEA 29604    1
THEA 11844    1
Name: count, Length: 149, dtype: int64

**Doing Some Data Manipulation**

Having pulled and checked the data out - we can see that there are some courses with singular instances, and others with multiple (up to 8 - the max possible in our dataset). While we will maintain the courses with singular instances, we also need to consider the "default" state of the line chart. I considered making the default state the first course on the list, but after some consideration I think making a "total" option for each department would be better. This provides the overview for the unit, and then allows for increased granularity over time. Additionally, since I am planning on using the percentages for the line chart, I will want to make columns for those as well!

In [516]:
#Making Department Totals Pt. 1 (Identifying Departments)
mtpct["Department"].unique()

<StringArray>
['THDN', 'MDJ', 'MUS', 'ARCH', 'VCD', 'ART', 'EMAT', 'FDM', 'CCI', 'COMM']
Length: 10, dtype: str

In [517]:
#Making Department Totals Pt. 2 (Group By Attempt 1)
mtpctdept1 = mtpct.groupby(["Department","Academic Period"])
mtpctdept1.head()
#It kind of worked - it seems to have grouped the courses together, not the total of everything. I'm going to try something else.

,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades
0,THEA 28661,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0,1.000000,100.00%
1,MDJ 11788,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0,1.000000,100.00%
2,DAN 22742,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0,1.000000,100.00%
3,THEA 14873,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0,1.000000,100.00%
4,MUS 20682,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0,1.000000,100.00%
...,...,...,...,...,...,...,...,...,...,...,...
677,EMAT 12670,202280,EMAT,CCI,EMAT,19.0,3.0,4.0,26.0,0.730769,73.08%
708,EMAT 26013,202580,EMAT,CCI,EMAT,42.0,8.0,9.0,59.0,0.711864,71.19%
728,EMAT 17213,202610,EMAT,CCI,EMAT,21.0,4.0,5.0,30.0,0.700000,70.00%
741,CCI 11716,202480,CCI,CCI,CCI,52.0,10.0,14.0,76.0,0.684211,68.42%


In [518]:
# Making Department Totals Pt. 3 (Removing Some Pieces)
# So this may be making it harder for me in the long run. The thought process is to cut out the course codes and passing grade counts.
# Former because it made the groupby not work, the latter because the counts would just add up.
mtpctdept = mtpct.drop(columns = ["Course Code", "# of Passing Grades", "% of Passing Grades"])
mtpctdept.head()

,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0
1,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0
2,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0
3,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0
4,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0


In [519]:
#Making Department Totals Pt. 4 (Group By Attempt 2)
mtpctdept2 = mtpctdept.groupby(["Subject", "Academic Period"])
mtpctdept2.head()
#So that didn't seem to exactly work either - let's try revisiting something else. I found an article that references uses .sum() (https://www.geeksforgeeks.org/python/pandas-groupby-and-sum/)

,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0
1,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0
2,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0
3,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0
4,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0
...,...,...,...,...,...,...,...,...
757,202480,ARTS,CotA,ART,38.0,8.0,13.0,59.0
759,202310,ARCS,CAED,ARCH,9.0,2.0,3.0,14.0
760,202480,ARCS,CAED,ARCH,7.0,2.0,2.0,11.0
769,202580,ARCS,CAED,ARCH,6.0,2.0,2.0,10.0


In [520]:
#Making Department Totals Pt. 5 (Group By Attempt 3)
mtpctdept3 = mtpctdept.groupby(["Subject", "Academic Period"]).sum()
mtpctdept3
#OK - we're getting closer! It did what I wanted, but also combined the colleges and departments together (so now it looks like CCICCICCICCI).
# It recommended calling out the specific columns I wanted to group instead. I'm going to see if putting them in a variable works

Subject College  \
Subject Academic Period                                      
AED     202280                                        CAED   
        202310                                        CAED   
        202380                                        CAED   
        202410                                        CAED   
        202480                                        CAED   
...                                                    ...   
VCD     202410              CCICCICCICCICCICCICCICCICCICCI   
        202480           CCICCICCICCICCICCICCICCICCICCICCI   
        202510                 CCICCICCICCICCICCICCICCICCI   
        202580                 CCICCICCICCICCICCICCICCICCI   
        202610                    CCICCICCICCICCICCICCICCI   

                                                Department  MT C or Higher  \
Subject Academic Period                                                      
AED     202280                                        ARCH            84.0   
        202310                                        ARCH             8.0   
        202380                                        ARCH            58.0   
        202410                                        ARCH            10.0   
        202480                                        ARCH            75.0   
...                                                    ...             ...   
VCD     202410              VCDVCDVCDVCDVCDVCDVCDVCDVCDVCD           548.0   
        202480           VCDVCDVCDVCDVCDVCDVCDVCDVCDVCDVCD           615.0   
        202510                 VCDVCDVCDVCDVCDVCDVCDVCDVCD           449.0   
        202580                 VCDVCDVCDVCDVCDVCDVCDVCDVCD           439.0   
        202610                    VCDVCDVCDVCDVCDVCDVCDVCD           353.0   

                         MT C-, D, F, W  MT Not Reported  Total Grades  
Subject Academic Period                                                 
AED     202280                      6.0             13.0         103.0  
        202310                      5.0              1.0          14.0  
        202380                     12.0              9.0          79.0  
        202410                      1.0              1.0          12.0  
        202480                     11.0              7.0          93.0  
...                                 ...              ...           ...  
VCD     202410                     66.0             65.0         679.0  
        202480                     90.0             73.0         778.0  
        202510                     59.0             59.0         567.0  
        202580                     65.0             54.0         558.0  
        202610                     54.0             27.0         434.0  

[135 rows x 6 columns]

In [521]:
#Making Department Totals Pt. 6 (Grade Columns Variable)
gradecols = ["MT C or Higher", "MT C-, D, F, W", "MT Not Reported", "Total Grades"]

In [522]:
#Making Department Totals Pt. 7 (Group By Attempt 4)
mtpctdept4 = mtpctdept.groupby(["Subject", "Academic Period"])[gradecols].sum()
mtpctdept4
#So it worked, just dropped the columns we actually needed. I think we just have to call out all the columns first, and then we can go with it from there :)

MT C or Higher  MT C-, D, F, W  MT Not Reported  \
Subject Academic Period                                                    
AED     202280                     84.0             6.0             13.0   
        202310                      8.0             5.0              1.0   
        202380                     58.0            12.0              9.0   
        202410                     10.0             1.0              1.0   
        202480                     75.0            11.0              7.0   
...                                 ...             ...              ...   
VCD     202410                    548.0            66.0             65.0   
        202480                    615.0            90.0             73.0   
        202510                    449.0            59.0             59.0   
        202580                    439.0            65.0             54.0   
        202610                    353.0            54.0             27.0   

                         Total Grades  
Subject Academic Period                
AED     202280                  103.0  
        202310                   14.0  
        202380                   79.0  
        202410                   12.0  
        202480                   93.0  
...                               ...  
VCD     202410                  679.0  
        202480                  778.0  
        202510                  567.0  
        202580                  558.0  
        202610                  434.0  

[135 rows x 4 columns]

In [523]:
#Making Department Totals Pt. 8 (Group By Attempt 5)
mtpctdept = mtpctdept.groupby(["Subject", "Academic Period", "Subject College", "Department"])[gradecols].sum().reset_index()
mtpctdept
#It worked! I tacked on the reset_index for this instance so each row has a subject column. Now that it has it, we can move ahead.

,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades
0,AED,202280,CAED,ARCH,84.0,6.0,13.0,103.0
1,AED,202310,CAED,ARCH,8.0,5.0,1.0,14.0
2,AED,202380,CAED,ARCH,58.0,12.0,9.0,79.0
3,AED,202410,CAED,ARCH,10.0,1.0,1.0,12.0
4,AED,202480,CAED,ARCH,75.0,11.0,7.0,93.0
...,...,...,...,...,...,...,...,...
130,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0
131,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0
132,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0
133,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0


*Making the Percent Columns*

In [524]:
#Making the Percent Columns Pt. 1 (Making the # Columns - Decimal)
mtpctdept["# of Passing Grades"] = mtpctdept["MT C or Higher"]/mtpctdept["Total Grades"]
mtpctdept["# of Failing Grades"] = mtpctdept["MT C-, D, F, W"]/mtpctdept["Total Grades"]
mtpctdept["# of Unreported Grades"] = mtpctdept["MT Not Reported"]/mtpctdept["Total Grades"]
mtpctdept

,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades
0,AED,202280,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,0.058252,0.126214
1,AED,202310,CAED,ARCH,8.0,5.0,1.0,14.0,0.571429,0.357143,0.071429
2,AED,202380,CAED,ARCH,58.0,12.0,9.0,79.0,0.734177,0.151899,0.113924
3,AED,202410,CAED,ARCH,10.0,1.0,1.0,12.0,0.833333,0.083333,0.083333
4,AED,202480,CAED,ARCH,75.0,11.0,7.0,93.0,0.806452,0.118280,0.075269
...,...,...,...,...,...,...,...,...,...,...,...
130,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0,0.807069,0.097202,0.095729
131,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0,0.790488,0.115681,0.093830
132,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0,0.791887,0.104056,0.104056
133,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0,0.786738,0.116487,0.096774


In [525]:
#Making the Percent Columns Pt. 2 (Making the % Columns)
mtpctdept["% of Passing Grades"] = mtpctdept["# of Passing Grades"].apply("{:.2%}".format)
mtpctdept["% of Failing Grades"] = mtpctdept["# of Failing Grades"].apply("{:.2%}".format)
mtpctdept["% of Unreported Grades"] = mtpctdept["# of Unreported Grades"].apply("{:.2%}".format)
mtpctdept

,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades,% of Passing Grades,% of Failing Grades,% of Unreported Grades
0,AED,202280,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,0.058252,0.126214,81.55%,5.83%,12.62%
1,AED,202310,CAED,ARCH,8.0,5.0,1.0,14.0,0.571429,0.357143,0.071429,57.14%,35.71%,7.14%
2,AED,202380,CAED,ARCH,58.0,12.0,9.0,79.0,0.734177,0.151899,0.113924,73.42%,15.19%,11.39%
3,AED,202410,CAED,ARCH,10.0,1.0,1.0,12.0,0.833333,0.083333,0.083333,83.33%,8.33%,8.33%
4,AED,202480,CAED,ARCH,75.0,11.0,7.0,93.0,0.806452,0.118280,0.075269,80.65%,11.83%,7.53%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0,0.807069,0.097202,0.095729,80.71%,9.72%,9.57%
131,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0,0.790488,0.115681,0.093830,79.05%,11.57%,9.38%
132,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0,0.791887,0.104056,0.104056,79.19%,10.41%,10.41%
133,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0,0.786738,0.116487,0.096774,78.67%,11.65%,9.68%


*Making the Total Courses Column*

In [526]:
#Making the Total Courses Column Pt. 1 (First Attempt)
#mtpctdept["Course Code"] = (mtpctdept["Subject"] & " Term Total")
#mtpctdept
#This one errored out - I was using the wrong logic and needed to use the + icon instead

In [527]:
#Making the Total Courses Column Pt. 2 (Second Attempt)
mtpctdept["Course Code"] = mtpctdept["Subject"] + " - Term Total"
mtpctdept

,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,# of Failing Grades,# of Unreported Grades,% of Passing Grades,% of Failing Grades,% of Unreported Grades,Course Code
0,AED,202280,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,0.058252,0.126214,81.55%,5.83%,12.62%,AED - Term Total
1,AED,202310,CAED,ARCH,8.0,5.0,1.0,14.0,0.571429,0.357143,0.071429,57.14%,35.71%,7.14%,AED - Term Total
2,AED,202380,CAED,ARCH,58.0,12.0,9.0,79.0,0.734177,0.151899,0.113924,73.42%,15.19%,11.39%,AED - Term Total
3,AED,202410,CAED,ARCH,10.0,1.0,1.0,12.0,0.833333,0.083333,0.083333,83.33%,8.33%,8.33%,AED - Term Total
4,AED,202480,CAED,ARCH,75.0,11.0,7.0,93.0,0.806452,0.118280,0.075269,80.65%,11.83%,7.53%,AED - Term Total
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0,0.807069,0.097202,0.095729,80.71%,9.72%,9.57%,VCD - Term Total
131,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0,0.790488,0.115681,0.093830,79.05%,11.57%,9.38%,VCD - Term Total
132,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0,0.791887,0.104056,0.104056,79.19%,10.41%,10.41%,VCD - Term Total
133,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0,0.786738,0.116487,0.096774,78.67%,11.65%,9.68%,VCD - Term Total


In [528]:
#Reordering the Columns
mtpctdept = mtpctdept[["Course Code", "Subject", "Academic Period", "Subject College", "Department", "MT C or Higher",
                       "MT C-, D, F, W", "MT Not Reported", "Total Grades", "# of Passing Grades", "% of Passing Grades",
                       "# of Failing Grades", "% of Failing Grades", "# of Unreported Grades", "% of Unreported Grades"]]
mtpctdept

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
0,AED - Term Total,AED,202280,CAED,ARCH,84.0,6.0,13.0,103.0,0.815534,81.55%,0.058252,5.83%,0.126214,12.62%
1,AED - Term Total,AED,202310,CAED,ARCH,8.0,5.0,1.0,14.0,0.571429,57.14%,0.357143,35.71%,0.071429,7.14%
2,AED - Term Total,AED,202380,CAED,ARCH,58.0,12.0,9.0,79.0,0.734177,73.42%,0.151899,15.19%,0.113924,11.39%
3,AED - Term Total,AED,202410,CAED,ARCH,10.0,1.0,1.0,12.0,0.833333,83.33%,0.083333,8.33%,0.083333,8.33%
4,AED - Term Total,AED,202480,CAED,ARCH,75.0,11.0,7.0,93.0,0.806452,80.65%,0.118280,11.83%,0.075269,7.53%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,VCD - Term Total,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0,0.807069,80.71%,0.097202,9.72%,0.095729,9.57%
131,VCD - Term Total,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0,0.790488,79.05%,0.115681,11.57%,0.093830,9.38%
132,VCD - Term Total,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0,0.791887,79.19%,0.104056,10.41%,0.104056,10.41%
133,VCD - Term Total,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0,0.786738,78.67%,0.116487,11.65%,0.096774,9.68%


*Revisiting the "Full" Dataset*

This is going back to the first dataset and adjusting the formatting so it aligns with the "Term Total" datasets. Mostly just copying and pasting the code.

In [529]:
#Making the Full Percent Columns Pt. 1 (Making the # Columns - Decimal)
mtpct["# of Failing Grades"] = mtpct["MT C-, D, F, W"]/mtpct["Total Grades"]
mtpct["# of Unreported Grades"] = mtpct["MT Not Reported"]/mtpct["Total Grades"]
mtpct.head()

,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,# of Unreported Grades
0,THEA 28661,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0,1.0,100.00%,0.0,0.0
1,MDJ 11788,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0,1.0,100.00%,0.0,0.0
2,DAN 22742,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.0
3,THEA 14873,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0,1.0,100.00%,0.0,0.0
4,MUS 20682,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.0


In [530]:
#Making the Full Percent Columns Pt. 2 (Making the % Columns)
mtpct["% of Failing Grades"] = mtpct["# of Failing Grades"].apply("{:.2%}".format)
mtpct["% of Unreported Grades"] = mtpct["# of Unreported Grades"].apply("{:.2%}".format)
mtpct.head()

,Course Code,Academic Period,Subject,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,# of Unreported Grades,% of Failing Grades,% of Unreported Grades
0,THEA 28661,202410,THEA,CotA,THDN,5.0,0.0,0.0,5.0,1.0,100.00%,0.0,0.0,0.00%,0.00%
1,MDJ 11788,202380,MDJ,CCI,MDJ,19.0,0.0,0.0,19.0,1.0,100.00%,0.0,0.0,0.00%,0.00%
2,DAN 22742,202480,DAN,CotA,THDN,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.0,0.00%,0.00%
3,THEA 14873,202480,THEA,CotA,THDN,11.0,0.0,0.0,11.0,1.0,100.00%,0.0,0.0,0.00%,0.00%
4,MUS 20682,202480,MUS,CotA,MUS,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.0,0.00%,0.00%


In [531]:
#Reording the Full Dataset
mtpct = mtpct[["Course Code", "Subject", "Academic Period", "Subject College", "Department", "MT C or Higher",
                       "MT C-, D, F, W", "MT Not Reported", "Total Grades", "# of Passing Grades", "% of Passing Grades",
                       "# of Failing Grades", "% of Failing Grades", "# of Unreported Grades", "% of Unreported Grades"]]
mtpct.head()

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
0,THEA 28661,THEA,202410,CotA,THDN,5.0,0.0,0.0,5.0,1.0,100.00%,0.0,0.00%,0.0,0.00%
1,MDJ 11788,MDJ,202380,CCI,MDJ,19.0,0.0,0.0,19.0,1.0,100.00%,0.0,0.00%,0.0,0.00%
2,DAN 22742,DAN,202480,CotA,THDN,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.00%,0.0,0.00%
3,THEA 14873,THEA,202480,CotA,THDN,11.0,0.0,0.0,11.0,1.0,100.00%,0.0,0.00%,0.0,0.00%
4,MUS 20682,MUS,202480,CotA,MUS,9.0,0.0,0.0,9.0,1.0,100.00%,0.0,0.00%,0.0,0.00%


*Combining the Two Datasets*

In [532]:
#Combining the Datasets
mtpctfull = pd.concat([mtpct, mtpctdept])
mtpctfull
#I got this idea from the pandas cheat sheet from the start of the term - this creates the "complete" dataset we can reference

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
0,THEA 28661,THEA,202410,CotA,THDN,5.0,0.0,0.0,5.0,1.000000,100.00%,0.000000,0.00%,0.000000,0.00%
1,MDJ 11788,MDJ,202380,CCI,MDJ,19.0,0.0,0.0,19.0,1.000000,100.00%,0.000000,0.00%,0.000000,0.00%
2,DAN 22742,DAN,202480,CotA,THDN,9.0,0.0,0.0,9.0,1.000000,100.00%,0.000000,0.00%,0.000000,0.00%
3,THEA 14873,THEA,202480,CotA,THDN,11.0,0.0,0.0,11.0,1.000000,100.00%,0.000000,0.00%,0.000000,0.00%
4,MUS 20682,MUS,202480,CotA,MUS,9.0,0.0,0.0,9.0,1.000000,100.00%,0.000000,0.00%,0.000000,0.00%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,VCD - Term Total,VCD,202410,CCI,VCD,548.0,66.0,65.0,679.0,0.807069,80.71%,0.097202,9.72%,0.095729,9.57%
131,VCD - Term Total,VCD,202480,CCI,VCD,615.0,90.0,73.0,778.0,0.790488,79.05%,0.115681,11.57%,0.093830,9.38%
132,VCD - Term Total,VCD,202510,CCI,VCD,449.0,59.0,59.0,567.0,0.791887,79.19%,0.104056,10.41%,0.104056,10.41%
133,VCD - Term Total,VCD,202580,CCI,VCD,439.0,65.0,54.0,558.0,0.786738,78.67%,0.116487,11.65%,0.096774,9.68%


**Making the % Line Chart**

The dataset has been updated to meet our needs! I like the idea of having a "Total" option. I'm contemplating adding it onto the final version of the table. Something I've contemplated is, instead of sorting by the pass rate, sorting by the Course Code. This logic would have a more natural flow, and (ideally) should end with the Term Total before moving onto the next subject (when viewing the full table w/o filters). I'm going to keep this in the back of my mind for the final version of the project.

For now, we're going to move onto creating the line chart. I was contemplating using Seaborn, but since we need filters, we're going back to the old reliable that is plotly.

In [533]:
#Making a Simple Line Chart
#To start, I just want to make a simple chart and plug some data in to make sure it can work and get some ideas of what needs done
fig1 = go.Figure(data = go.Scatter(
    x = mtpctfull["Academic Period"],
    y = mtpctfull["% of Passing Grades"]
))

fig1.show()

What a beautiful mess! This tells me a few things that I need to do:
1. This will work - just need to add a few bits and bobs to make it functioning (legend, the failing and nonreported rates, the total number of grades, a title, dropdown menu)
2. I will need to transition the Academic Period data to categorical (I think)
3. I will need to flip the Y axis so 100% is the top and 0% is the bottom

Knowing the limitations of PowerBI ahead of time also adds an extra layer to this - **I need to make this in a way that it is easy to duplicate the code for each subject/department**. I believe the best way to approach this is having the subject/area defined first (i.e. currentmajor = MAJ).

*Academic Period Data to Categorical*

Since the other items are tied to manipulating the line chart, I'm going to do the data type conversion first.

In [534]:
#Making categorical Term Data Pt. 1 (Checking Types)
mtpctfull.dtypes
#It is an integer, so we will need to convert it to a category!

Course Code                   str
Subject                       str
Academic Period             int64
Subject College               str
Department                    str
MT C or Higher            float64
MT C-, D, F, W            float64
MT Not Reported           float64
Total Grades              float64
# of Passing Grades       float64
% of Passing Grades           str
# of Failing Grades       float64
% of Failing Grades           str
# of Unreported Grades    float64
% of Unreported Grades        str
dtype: object

In [535]:
#Making Categorical Term Data Pt. 2 (Converting to Categories)
mtpctfull["Academic Period"] = mtpctfull["Academic Period"].astype("category")
mtpctfull.dtypes

Course Code                    str
Subject                        str
Academic Period           category
Subject College                str
Department                     str
MT C or Higher             float64
MT C-, D, F, W             float64
MT Not Reported            float64
Total Grades               float64
# of Passing Grades        float64
% of Passing Grades            str
# of Failing Grades        float64
% of Failing Grades            str
# of Unreported Grades     float64
% of Unreported Grades         str
dtype: object

*Making the Line Chart Variables*

The first thing we'll want to do is make all of the variables for each unit. This will vary slightly depending on if the unit has no schools (only the college) or schools/departments (so breakdown into different levels based on subject). These variables will be used for specifying what courses each line chart will look at.

In [536]:
#Making Line Chart Variables Pt. 1 (CAED Subject Variables)
#CAED is the only college of the three that does not have a school model
aedmtpct = mtpctfull[mtpctfull["Subject"] == "AED"]
archmtpct = mtpctfull[mtpctfull["Subject"] == "ARCH"]
cmgtmtpct = mtpctfull[mtpctfull["Subject"] == "CMGT"]
idmtpct = mtpctfull[mtpctfull["Subject"] == "ID"]

In [537]:
#Making Line Chart Variables Pt. 2 (CCI SSchool Variables)
#CCI has schools, so we need to make a variable for each school
ccimtpct = mtpctfull[mtpctfull["Department"] == "CCI"]
commmtpct = mtpctfull[mtpctfull["Department"] == "COMM"]
ematmtpct = mtpctfull[mtpctfull["Department"] == "EMAT"]
mdjmtpct = mtpctfull[mtpctfull["Department"] == "MDJ"]
vcdmtpct = mtpctfull[mtpctfull["Department"] == "VCD"]

In [538]:
#Making Line Chart Variables Pt. 3 (CotA School Variables)
#CotA has schools, so we need to make a variable for each school
artmtpct = mtpctfull[mtpctfull["Department"] == "ART"]
fdmmtpct = mtpctfull[mtpctfull["Department"] == "FDM"]
musmtpct = mtpctfull[mtpctfull["Department"] == "MUS"]
thdnmtpct = mtpctfull[mtpctfull["Department"] == "THDN"]

*Making the Line Chart*

Now that we have defined the variables for the charts, we can start making them. I'm going to just use the aedmtpct variable for experimenting with charts. Once I've made the "final" version, I'll run through and make updated variables for them all.

In [539]:
#Making the Line Chart Pt. 1 (First Attempt)
fig2 = go.Figure(data = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
))

fig2.show()

So it is still messy, but much easier to read. I think the next step to resolve is getting the x and y axis to play along.

In [540]:
#Making the Line Chart Pt. 2 (X and Y Working)
fig3 = go.Figure(data = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
))

fig3.update_layout(
    xaxis = dict(
        type = "category" #I did some looking online and this seems to be the "quick fix" - if this is true, can probably skip the category change earlier in the code
    ),
    yaxis = dict(
        autorange = True, #After some research - it seems like this happens when Plotly misinterprets the data range. This will force it to work
        range = [0,100] #Formally defining the range
    )
)

fig3.show()

So it kind of worked. I got the set terms, but they are trying to do them in a set order and the Y axis didn't flip itself. There was an alternative term to use (reversed) that may fix it, so going to try that. First

In [541]:
fig4 = go.Figure(data = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
))

fig4.update_layout(
    xaxis = dict(
        type = "category",
        categoryorder = "array", #This is trying to dictate they should follow an order
        categoryarray = [202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610] #This is specifying the order
    ),
    yaxis = dict(
        autorange = "reversed", #This is the plan b
        range = [0,100] #Formally defining the range
    )
)

fig4.show()

So good news, bad news. Good news - the y axis flipped and the x axis is ordered! Bad news - I was referencing my notes and realized that this is actually one line bouncing back and forth. My notes say we need to sort that column, so we're going to try that (and re-run the variables below so those are the "new" ones).

In [542]:
#Sorting the Periods Pt. 1 (Sorting Values)
mtpctfull = mtpctfull.sort_values("Academic Period")
mtpctfull

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
408,THEA 10881,THEA,202280,CotA,THDN,327.0,47.0,41.0,415.0,0.787952,78.80%,0.113253,11.33%,0.098795,9.88%
56,CMGT - Term Total,CMGT,202280,CAED,ARCH,156.0,21.0,25.0,202.0,0.772277,77.23%,0.103960,10.40%,0.123762,12.38%
198,FDM 12681,FDM,202280,CotA,FDM,75.0,10.0,6.0,91.0,0.824176,82.42%,0.109890,10.99%,0.065934,6.59%
200,ARCH 16841,ARCH,202280,CAED,ARCH,313.0,42.0,25.0,380.0,0.823684,82.37%,0.110526,11.05%,0.065789,6.58%
631,MDJ 22796,MDJ,202280,CCI,MDJ,44.0,10.0,5.0,59.0,0.745763,74.58%,0.169492,16.95%,0.084746,8.47%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544,FDM 25693,FDM,202610,CotA,FDM,26.0,2.0,6.0,34.0,0.764706,76.47%,0.058824,5.88%,0.176471,17.65%
549,COMM 21241,COMM,202610,CCI,COMM,42.0,9.0,4.0,55.0,0.763636,76.36%,0.163636,16.36%,0.072727,7.27%
571,EMAT 26013,EMAT,202610,CCI,EMAT,44.0,7.0,7.0,58.0,0.758621,75.86%,0.120690,12.07%,0.120690,12.07%
459,FDM 24923,FDM,202610,CotA,FDM,120.0,24.0,10.0,154.0,0.779221,77.92%,0.155844,15.58%,0.064935,6.49%


In [543]:
#Sorting the Periods Pt. 2 (Redefining variables)
aedmtpct = mtpctfull[mtpctfull["Subject"] == "AED"]
archmtpct = mtpctfull[mtpctfull["Subject"] == "ARCH"]
cmgtmtpct = mtpctfull[mtpctfull["Subject"] == "CMGT"]
idmtpct = mtpctfull[mtpctfull["Subject"] == "ID"]
ccimtpct = mtpctfull[mtpctfull["Department"] == "CCI"]
commmtpct = mtpctfull[mtpctfull["Department"] == "COMM"]
ematmtpct = mtpctfull[mtpctfull["Department"] == "EMAT"]
mdjmtpct = mtpctfull[mtpctfull["Department"] == "MDJ"]
vcdmtpct = mtpctfull[mtpctfull["Department"] == "VCD"]
artmtpct = mtpctfull[mtpctfull["Department"] == "ART"]
fdmmtpct = mtpctfull[mtpctfull["Department"] == "FDM"]
musmtpct = mtpctfull[mtpctfull["Department"] == "MUS"]
thdnmtpct = mtpctfull[mtpctfull["Department"] == "THDN"]

In [544]:
#Rerunning the figure
fig5 = go.Figure(data = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
))

fig5.update_layout(
    xaxis = dict(
        type = "category", 
    ),
    yaxis = dict(
        autorange = "reversed", 
        range = [0,100] 
    )
)

fig5.show()

So it worked, but the Y axis is being funky. It was at this point I scrolled up to the dtypes earlier and realized that my %'s were not being treated as numerical values. Going to change that.

In [545]:
#Converting the % to numbers Pt. 1 (First Attempt)
mtpctfull["% of Passing Grades"] = pd.to_numeric(mtpctfull["% of Passing Grades"])
mtpctfull["% of Failing Grades"] = pd.to_numeric(mtpctfull["% of Failing Grades"])
mtpctfull["% of Unreported Grades"] = pd.to_numeric(mtpctfull["% of Unreported Grades"])
#Gah - the % is throwing a wrench into it. It looks nice for the table, but we probably should cut it here. going to strip it.

ValueError: Unable to parse string "78.80%" at position 0

In [546]:
#Converting the % to numbers Pt. 2 (Removing the %)
mtpctfull["% of Passing Grades"] = mtpctfull["% of Passing Grades"].str.strip("%") #Just rewriting the columns with the number minus the %
mtpctfull["% of Failing Grades"] = mtpctfull["% of Failing Grades"].str.strip("%")
mtpctfull["% of Unreported Grades"] = mtpctfull["% of Unreported Grades"].str.strip("%")
mtpctfull.head()

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
408,THEA 10881,THEA,202280,CotA,THDN,327.0,47.0,41.0,415.0,0.787952,78.80,0.113253,11.33,0.098795,9.88
56,CMGT - Term Total,CMGT,202280,CAED,ARCH,156.0,21.0,25.0,202.0,0.772277,77.23,0.103960,10.40,0.123762,12.38
198,FDM 12681,FDM,202280,CotA,FDM,75.0,10.0,6.0,91.0,0.824176,82.42,0.109890,10.99,0.065934,6.59
200,ARCH 16841,ARCH,202280,CAED,ARCH,313.0,42.0,25.0,380.0,0.823684,82.37,0.110526,11.05,0.065789,6.58
631,MDJ 22796,MDJ,202280,CCI,MDJ,44.0,10.0,5.0,59.0,0.745763,74.58,0.169492,16.95,0.084746,8.47


In [547]:
#Converting the % to numbers Pt. 3 (Making them numbers)
mtpctfull["% of Passing Grades"] = pd.to_numeric(mtpctfull["% of Passing Grades"])
mtpctfull["% of Failing Grades"] = pd.to_numeric(mtpctfull["% of Failing Grades"])
mtpctfull["% of Unreported Grades"] = pd.to_numeric(mtpctfull["% of Unreported Grades"])
mtpctfull.dtypes

Course Code                    str
Subject                        str
Academic Period           category
Subject College                str
Department                     str
MT C or Higher             float64
MT C-, D, F, W             float64
MT Not Reported            float64
Total Grades               float64
# of Passing Grades        float64
% of Passing Grades        float64
# of Failing Grades        float64
% of Failing Grades        float64
# of Unreported Grades     float64
% of Unreported Grades     float64
dtype: object

In [548]:
#Converting the % to numbers Pt. 4 (Redefining variables)
aedmtpct = mtpctfull[mtpctfull["Subject"] == "AED"]
archmtpct = mtpctfull[mtpctfull["Subject"] == "ARCH"]
cmgtmtpct = mtpctfull[mtpctfull["Subject"] == "CMGT"]
idmtpct = mtpctfull[mtpctfull["Subject"] == "ID"]
ccimtpct = mtpctfull[mtpctfull["Department"] == "CCI"]
commmtpct = mtpctfull[mtpctfull["Department"] == "COMM"]
ematmtpct = mtpctfull[mtpctfull["Department"] == "EMAT"]
mdjmtpct = mtpctfull[mtpctfull["Department"] == "MDJ"]
vcdmtpct = mtpctfull[mtpctfull["Department"] == "VCD"]
artmtpct = mtpctfull[mtpctfull["Department"] == "ART"]
fdmmtpct = mtpctfull[mtpctfull["Department"] == "FDM"]
musmtpct = mtpctfull[mtpctfull["Department"] == "MUS"]
thdnmtpct = mtpctfull[mtpctfull["Department"] == "THDN"]

In [549]:
#Rerunning the figure
fig6 = go.Figure(data = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
))

fig6.update_layout(
    xaxis = dict(
        type = "category",
    ),
    yaxis = dict(
        range = [0,100] 
    )
)

fig6.show()

OK! Now we've got something we can work with. I think the next step is to try and do the traces for the other rates.

In [550]:
#Attempting to add traces
aedpasstrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"]
)

aedfailtrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Failing Grades"]
)

aednatrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Unreported Grades"]
)

layout = go.Layout(
    xaxis = dict(
        type = "category", 
    ),
    yaxis = dict(
        range = [0,100] 
    )
)

fig7 = go.Figure(data = [aedpasstrace, aedfailtrace, aednatrace], layout = layout)

fig7.show()

Now we've got the skeleton down. I think we can shift gears from getting it to show the data to making it look nice. Up next, we're going to do the following:
- Name the traces
- Color code them fitting colors (Pass = green, Fail = Red, Unreported = Gray)
- Add a title
- Add a small buffer to the bottom and top range - this way it doesn't look like they disappear and are fully visible

In [551]:
#Making the Line Chart Look Nice
aedpasstrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Passing Grades"],
    name = "Pass Grade Rate", #labeling the line
    line = dict(color = "green", width = 2) #deciding the color of the line
)

aedfailtrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Failing Grades"],
    name = "Fail Grade Rate",
    line = dict(color = "red", width = 2)
)

aednatrace = go.Scatter(
    x = aedmtpct["Academic Period"],
    y = aedmtpct["% of Unreported Grades"],
    name = "Unreported Grade Rate",
    line = dict(color = "gray", width = 2)
)

layout = go.Layout(
    xaxis = dict(
        type = "category", 
    ),
    yaxis = dict(
        range = [-5,105] 
    ),
    title=dict(text= "Grade Rates Over Time", #adding the title - using the logic from the other tables to customize this (for now - may tweak more later)
        font_weight = "bold", 
        xanchor = "center",
        x = .5),
)

fig7 = go.Figure(data = [aedpasstrace, aedfailtrace, aednatrace], layout = layout)

fig7.show()

I think that looks much better - calls in biological relevance with the colors, has a (very rough) working title, and better shows when a course will have 0% or 100%. Something that I'd like to add next are the buttons. This will allow filtering by the subjects to show how the rates fluctuated over time. To accomplish this, we will need to write loops for each rate (pass, fail, and unreported rates).

In [552]:
#Making the Looped Version (Pt. 1 Working but not really)

courses = aedmtpct["Course Code"].unique()
coursebuttons = []

for i, course in enumerate(courses): #saying go through every instance of courses above to fill in the course variable
    coursepct = aedmtpct[aedmtpct["Course Code"] == course] #saying that we should make a new data frame for each course

    is_visible = (i == 0) #saying only have the first one visible

    aedpasstrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "Pass Grade Rate",
        line = dict(color = "green", width = 2)
    )

    aedfailtrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "Fail Grade Rate",
        line = dict(color = "red", width = 2)
    )

    aednatrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "Unreported Grade Rate",
        line = dict(color = "gray", width = 2)
    )

    #So the below logic was a bit of trial and error - trying to make an equation that is saying make them not all visible, but here is logic to make the one visible
    #Got the idea from https://plotly.com/python/dropdowns/ when looking at the dropdown logic
    visibility = [False] * (len(courses) * 3) #This code is saying that visibility (by default) is all false (the len(courses) * 3 is saying make all of the traces hidden)
    visibility[i*3: i*3 + 3] = [True, True, True] #This piece is saying whatever the current trace is should be all true (and thus, visible!)

    buttons = dict(
        label = course,
        method = "update",
        args = {"visible": visibility} #Using the visibility from earlier - saying whatever the choice is, use that one! It'll overwrite it later
    )

    coursebuttons.append(buttons)

    layout = go.Layout(
        xaxis = dict(
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105] 
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
    )

    ratelines = go.Figure(data = [aedpasstrace, aedfailtrace, aednatrace], layout = layout)

updatelayout = updatemenus = [dict(
        type = "dropdown",
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.1,
        buttons = coursebuttons)]

ratelines.show()

So no buttons. I looked over what I did and realized it is with the update layout. Going to rewrite so it works right - I also need to define visibility (forgot to do that in my original steps)

In [553]:
#Making the Looped Version Pt. 2 (figure issue)

courses = aedmtpct["Course Code"].unique()
coursebuttons = []

for i, course in enumerate(courses): 
    coursepct = aedmtpct[aedmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    aedpasstrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "Pass Grade Rate",
        line = dict(color = "green", width = 2),
        visible = is_visible
    )

    aedfailtrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "Fail Grade Rate",
        line = dict(color = "red", width = 2),
        visible = is_visible
    )

    aednatrace = go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "Unreported Grade Rate",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    )

    visibility = [False] * (len(courses) * 3)
    visibility[i*3: i*3 + 3] = [True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = {"visible": visibility}
    )

    coursebuttons.append(buttons)

    layout = go.Layout(
        xaxis = dict(
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105] 
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
    )

    ratelines = go.Figure(data = [aedpasstrace, aedfailtrace, aednatrace], layout = layout)

    ratelines.update_layout(
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.1,
            buttons = coursebuttons)])

ratelines.show()

ValueError: 
    Invalid value of type 'builtins.dict' received for the 'args' property of layout.updatemenu.button
        Received value: {'visible': [True, True, True, False, False, False]}

    The 'args' property is an info array that may be specified as:

    * a list or tuple of up to 3 elements where:
(0) The 'args[0]' property accepts values of any type
(1) The 'args[1]' property accepts values of any type
(2) The 'args[2]' property accepts values of any type

So now it is erroring out. I did some looking online, and it seems like the issue is tied to the location of the figure creation. Based on the plotly API, I need to define the figure outside the graph object, and then add the traces within the loop. What it seems to be doing this time is create the figure in the loop itself, and not adding the traces. (https://plotly.com/python-api-reference/generated/generated/plotly.graph_objects.Figure.add_traces.html)

In [554]:
#Making the Looped Version Pt. 3 (The working ver.)

courses = aedmtpct["Course Code"].unique()
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = aedmtpct[aedmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter( #Need to change this to "figure add trace" - this is what actually puts it in.
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "Pass Grade Rate",
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "Fail Grade Rate",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "Unreported Grade Rate",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    visibility = [False] * (len(courses) * 3)
    visibility[i*3: i*3 + 3] = [True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105] 
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.1,
            buttons = coursebuttons)])

ratelines.show()

There we go! So this time it did work, but there are a few smaller tweaks/things to note:
1. In hindsight, choosing AED as my guinea pig wasn't the best choice. It only has one course that qualifies for MT grades. Still, it works! What I may do is shift to another subject that I know has a lot of course options for MT grades to make sure the buttons work.
2. I'll need to do some tweaking with the button location. The legend makes the figure wider, so that throws it off a bit. I'll need to do a bit of tweaking.
3. The legend is important, but the location definitely needs moved. I'm wondering how it'd look if I moved it to be underneath the figure horizontally? I may also tweak the title. If it looks good, I could keep the button location the same.
4. Something that may be important to consider is the number of students in each term. Using the figure above as an example, 202310 was a bad year for MT grades. However, we need to ask how many students are in the class. If it was 50, that's not good. But if it was something like 10, it isn't *as* bad since there is a higher chance for the numbers to be off due to the smaller number of students. 
5. I want to experiment with the background a bit - I have an idea for the number of students that would make the gray background not as good, but maybe adjusting the lines would be better.

In [555]:
#Adjusting Aspects Pt. 1 (Redefining a line labels and moving the legend)
courses = mdjmtpct["Course Code"].unique() #changing the subject to something with more course choices
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher", #I'm changing these to be more specific with what grades constitute the rate. This also aligns with what is visible on the table for consistency
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    visibility = [False] * (len(courses) * 3)
    visibility[i*3: i*3 + 3] = [True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105] 
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.1,
            buttons = coursebuttons)],
        legend = dict(
            xanchor = "center", #going to try the same logic as the buttons, but instead of top for the y axis, using bottom instead
            x = .5,
            yanchor = "bottom", 
            y = 1.1,
            orientation = "h" #I did some looking - this can define which way the legend goes - h = horizontal
        )
        )

ratelines.show()

So that worked and the logic is sound, but I'll need to experiment with moving the legend some. I'll also need to tweak the button height a bit to get it right. To save you the pain of seeing the legend and buttons moved a lot, below is the final code with the locations correct (there was a lot of trial and error using negative numbers and tiny tweaks to get it just right)

In [556]:
#Moving the Legend
courses = mdjmtpct["Course Code"].unique() #changing the subject to something with more course choices
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher", #I'm changing these to be more specific with what grades constitute the rate. This also aligns with what is visible on the table for consistency
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    visibility = [False] * (len(courses) * 3)
    visibility[i*3: i*3 + 3] = [True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105] 
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.15,
            buttons = coursebuttons)],
        legend = dict(
            xanchor = "center", #going to try the same logic as the buttons, but instead of top for the y axis, using bottom instead
            x = .5,
            yanchor = "bottom", 
            y = -.25,
            orientation = "h" #I did some looking - this can define which way the legend goes - h = horizontal
        )
        )

ratelines.show()

Now that looks nicer - more direct with what constitutes each button, aligns with the other figure, and the items are in a logical place.

So the next hurdle I need to confront is the number of students. This one is tricky, because we're pulling a different value and changing it based on the point. Something I thought about doing was adding to the hover text the number of students that earned that grade. I think that's a fine idea, but I want something that is more "at a glance" and gives you the total number of students in each term. At this point I consulted Gemini to see what recommendations it has. I provided my code so it had an understanding of the logic and posed the following prompt:

*So I would like to add the total number of students enrolled in each term to my figure. This number would need to fluctuate to reflect the number in the courses. How would I do that?*

What it proposed is adding a bar chart with low opacity to the figure that calculates the total number of students in a term. This would involve adjusting a secondary y axis as well so it can be tracked. I'm going to try it out and see what it looks like. Something it pointed out is needing to label the y axes (which I haven't done yet - so that'll be nice to have!).

In [557]:
#Attempting to Add Counts Pt. 1 (Adding the Counts)
courses = mdjmtpct["Course Code"].unique() #changing the subject to something with more course choices
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher", #I'm changing these to be more specific with what grades constitute the rate. This also aligns with what is visible on the table for consistency
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    #This is the code Gemini proposed (just this section)
    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0)    
    ))

    visibility = [False] * (len(courses) * 4) #need to bump this up now that we've got four figures
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
                title = "Semester",
                type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = "Percentage (%)"
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False #this is an extra piece it proposed - it removes the gride from this axis so it doesn't look as clutted
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.15,
            buttons = coursebuttons)],
        legend = dict(
            xanchor = "center", #going to try the same logic as the buttons, but instead of top for the y axis, using bottom instead
            x = .5,
            yanchor = "bottom", 
            y = -.25,
            orientation = "h" #I did some looking - this can define which way the legend goes - h = horizontal
        )
        )

ratelines.show()

So I'm not sure how much I like that. On one hand, it shows the total number of students and it isn't terribly intrusive to the main chart. However, the second y-axis and title isn't that great, and I feel the legend helps clarify that. Let me try to get rid of that first and see what happens

In [558]:
#Attempting to Add Counts Pt. 2 (Removing the Y axis)
courses = mdjmtpct["Course Code"].unique() #changing the subject to something with more course choices
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher", #I'm changing these to be more specific with what grades constitute the rate. This also aligns with what is visible on the table for consistency
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0)    
    ))

    visibility = [False] * (len(courses) * 4) #need to bump this up now that we've got four figures
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
                title = "Semester",
                type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = "Percentage (%)"
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.15,
            buttons = coursebuttons)],
        legend = dict(
            xanchor = "center", #going to try the same logic as the buttons, but instead of top for the y axis, using bottom instead
            x = .5,
            yanchor = "bottom", 
            y = -.25,
            orientation = "h" #I did some looking - this can define which way the legend goes - h = horizontal
        )
        )

ratelines.show()

OK - so getting rid of it outright does not work at all. It ends up hiding the lines, plus the main y axis gets cluttered. I'm going to try using the visible property - that may hide it for us

In [559]:
#Attempting to Add Counts Pt. 1 (Adding the Counts)
courses = mdjmtpct["Course Code"].unique() #changing the subject to something with more course choices
coursebuttons = []

ratelines = go.Figure() #moving this out here

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher", #I'm changing these to be more specific with what grades constitute the rate. This also aligns with what is visible on the table for consistency
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0)    
    ))

    visibility = [False] * (len(courses) * 4) #need to bump this up now that we've got four figures
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout( #I also would need to move the layout here now - since we are not definining it in go.Figure
                xaxis = dict(
                title = "Semester",
                type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = "Percentage (%)"
        ),
        yaxis2 = dict(#This is the criteria for the other axis
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False #saying to hide this one
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.15,
            buttons = coursebuttons)],
        legend = dict(
            xanchor = "center", #going to try the same logic as the buttons, but instead of top for the y axis, using bottom instead
            x = .5,
            yanchor = "bottom", 
            y = -.25,
            orientation = "h" #I did some looking - this can define which way the legend goes - h = horizontal
        )
        )

ratelines.show()

There we go - so much nicer. However, now that we have this extra layer, we need to do some work with the layout again. Mostly small tweaks, but I'll highlight what changed below

In [560]:
#Tweaking the Visualizations
courses = mdjmtpct["Course Code"].unique() 
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0)    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So now that I've formatted it a bit better, I want to tweak the visualization one last time before putting it to bed. I want to do the following:
- Change the hover text for each option - I want it to show the term, percent, and number of students that are being counted. We covered this in the second plotly group, so going to take their code and tweak it for my needs.
- Another piece I want to talk to Gemini about - I want the default state for these to be the "Total" Grade Rates - not a specific course. The prompt I engaged Gemini with was the following: *One last piece I want to do - in each of my subjects, I have an item that is Subject - Term Total. How could I set that to be the "default" state of my table? For our purposes, use MDJ - Term Total in this example: (Insert Code Here)*. It recommended sorting and forcing MDJ - Term Total to be first. In hindsight, I should have known to do this. However, the code it proposed was new to me, so I got to learn something new out of it!

In [561]:
#Adjusting the Sort
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 

    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "gray", width = 2),
        visible = is_visible
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0)    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

It worked - Nice! Now I'm going to add the unique hover text. Since each block is a different rate, I'll have to define the hover template in each trace.

In [562]:
#Adding the hover text
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

I think I'm ready to put a pin in this one for now. I want to show this to my peers during the peer critique and get their feedback. The main things I want to get is the following:
- Thoughts on the look of the visualization - do you like the placement of things like the legend, buttons, title, etc.
- What are thoughts on the bar chart piece - do you think it is necessary information or not?
- Does the hovertemplate feel a bit overkill?

After the meeting on 4/23, this is the responses to the above questions:
- They liked the look of it - recommended keeping in mind size of things when adjusting visualizations in the future
- They were split on the bar chart. They saw the logic (helps explain some data a bit better), but didn't notice the bar chart until I pointed it out. They also noted that, when hovering over things, it would jump between the two which could be disorienting. They recommended making the marker size variable based on the number of students
- They liked the hover template and felt it helped explain a lot

I'm going to experiment with the size variable. I think that isn't too intrusive and could be a good way of showing the number of values present!

In [563]:
#Adjusting marker size based on # of Grades
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        marker = dict(size = coursepct["MT C or Higher"]),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So this kind of worked - but only for the specific classes. When you look at the Total, it just overtakes the whole screen. I'm going to reference the plotly docs (https://plotly.com/python-api-reference/generated/plotly.graph_objects.Scatter.html) for Scatter and the size section to see if I can manipulate it to be a set size. It seems like there is a combination of variables that I can reference:
- sizemin - sets the minimum size
- sizeref - is the scaling factor
- sizemode - sets the rules for how size is converted to pixels. 

I think the best thing to use may be sizemin. Let me see what I can find out.

In [564]:
#Adjusting marker size based on # of Grades
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        marker = dict(size = coursepct["MT C or Higher"], sizemode = "area"), #Gonna try area first
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        marker = dict(size = coursepct["MT C-, D, F, W"], sizemode = "area"),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        marker = dict(size = coursepct["MT Not Reported"], sizemode = "area"),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So I applied just sizemode and set it as area. That worked and helped make the size manageable, but for courses with low enrollment, the points become nonexistant. I think I'll need to call in that sizemin to say what the smallest possible size could be, and possibly sizeref to adjust the scaling of the dots.

In [565]:
#Adjusting marker size based on # of Grades
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        marker = dict(size = coursepct["MT C or Higher"], sizemode = "area", sizemin = 4, sizeref = 2), #getting more specific - you can set the minimum size and the way it factors
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        marker = dict(size = coursepct["MT C-, D, F, W"], sizemode = "area", sizemin = 4, sizeref = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        marker = dict(size = coursepct["MT Not Reported"], sizemode = "area", sizemin = 4, sizeref = 2),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So it helped make them more noticeable, but now they're all that size (unless the number starts flucutating and getting really big).

I'm going to switch from area to diameter and see if that does much better

In [566]:
#Adjusting marker size based on # of Grades
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        marker = dict(size = coursepct["MT C or Higher"], sizemode = "diameter", sizemin = 4, sizeref = 5),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        marker = dict(size = coursepct["MT C-, D, F, W"], sizemode = "diameter", sizemin = 4, sizeref = 5),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        marker = dict(size = coursepct["MT Not Reported"], sizemode = "diameter", sizemin = 4, sizeref = 5),
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Bar(
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So it seems to have reverted to the "old way" and made the dots huge. I think the manipulation to the size also throws off the other figures. It's almost like the Term Total is tanking the ability of size to scale the rest. Since size is so big and I have to add so many limitations to control it, it totally tanks the size of the non-term totals. 

I think I could add some logic that would change the size of only the Term Totals and use other logic for the non-term totals, but I am starting to second guess the value of size. As I was flipping through examples, there were many instances of the MT C- line crossing over with the Not Reported line, so the overlap is a negative. 

I want to try something experimental. Right now, the bar charts are the size. I'm wondering if I were to make a separate scatter plot that is fixed to the top of the chart (the looks like a #) that when you hover over it, it'll tell you what the size is. I'm going to do some tweaking with that 

In [567]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter( #This is shifting the logic to do the hashtags instead of the dots
        x = coursepct["Academic Period"],
        y = coursepct["Total Grades"],
        name = "Student Enrollment (n)",
        yaxis = "y2",
        marker = dict(color = "lightgray"),
        opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So that didn't work, but I had an idea. What if I just plugged in for the y axis 105 for the values across the board times the length of the terms? That would align with the y axis we already have, and should future proof it so it always has a dot for each spot. 

In [568]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = [105] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        #yaxis = "y2",
        marker = dict(color = "lightgray"),
        #opacity = .2,
        visible = is_visible,
        showlegend = (i == 0),
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{y} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        yaxis2 = dict(
            title = "Total Enrollment (n)",
            overlaying = "y",
            side = "right",
            showgrid = False,
            visible = False
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

It worked - but now the values are set at 105 in the hovertemplate. Confused, but we got the spirit! What I'm going to do is set a customdata to be the Total # of Grades Assigned - this way we get the right amounts. At the same time, I'm going to do a bit more editing. I'm going to bump the range up a touch (to 110), change the markers to be just the dots, and maybe style them to be something else. I'm thinking of pulling in a png of a person that you can hover over to view that, but let us see if we can get the rough version working first.

In [569]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "markers",
        x = coursepct["Academic Period"],
        y = [105] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        marker = dict(color = "blue", symbol = "hash-open", size = 10), #further defining the look using the plotly symbol guide
        visible = is_visible,
        showlegend = (i == 0),
        customdata = coursepct["Total Grades"],
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,110],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

So after all that it doesn't look as nice as I thought. It makes it seem like there are grades at 105% (which is impossible).

At this point I started looking into combo'ing plots together. There is the subplots option, but I think that'd complicate the logic with the term total. I did see something about using text instead of dots for a scatter plot, so I'm going to see if I can get that to work.

In [570]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text", #specifying the text mode
        text = coursepct["Total Grades"],
        x = coursepct["Academic Period"],
        y = [105] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        visible = is_visible,
        showlegend = (i == 0),
        customdata = coursepct["Total Grades"],
        hovertemplate = ("<b>Count of Students</b>"\
            "<br>Term: %{x}"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
            )    
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-5,110],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )]
        )

ratelines.show()

OK... so this is something we could maybe work with! I want to try experimenting a bit more here. There are a few things I don't like:
- I don't like the legend at the bottom. I'm thinking maybe cutting that from the legend, so need to add that.
- If I cut the legend, I'll need to add a label so it doesn't look like floating numbers. Maybe I can do another annotation (similar to the Course: one with the button)
- No longer need the hover text if I can get that to work.
- It doesn't look that great at the top of the figure. Maybe instead of putting it above, putting it below 0? I'd have to flip the range then.
- If I go below 0, I'd need to adjust the y axis too. Maybe I can hard fix the ticks so it goes 0 and up, and skips the negative numbers?

Lets try implemeneting some of this.

In [571]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"],
        x = coursepct["Academic Period"],
        y = [-10] * len(coursepct["Academic Period"]), #moving them to the bottom
        name = "Student Enrollment (n)",
        visible = is_visible,
        showlegend = (False),
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    ),
        dict( #adding another label for the text examples
            text = "Student Enrollment (n):",
            font_weight = "bold",
            showarrow = False,
            xanchor = "left",
            x = 0,
            yref = "paper",
            y = 0
        )]
        )

ratelines.show()

So I got most of it to work - since I did -15 for the range, the tick doesn't catch that, which is nice. The issue is the annotation. I'm going to toy with that a lot next to see if I can get that.

In [572]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"],
        x = coursepct["Academic Period"],
        y = [-10] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        visible = is_visible,
        showlegend = (False),
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    ),
        dict(
            text = "Enrollment:",
            font_size = 10,
            font_weight = "bold",
            showarrow = False,
            xref = "paper",
            x = -.05,
            yref = "y",
            y = -10
        )]
        )

ratelines.show()

So I got that to work, but I'm not a fan of it in front of the grid. I'd prefer if it looked like a table. I did some looking and you can introduce shapes to plotly figures, so that *may* be a fix. I'm going to toy with it and see if that works. 

For reference - this is the shape library I'm referencing is https://plotly.com/python/reference/layout/shapes/

In [573]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"],
        x = coursepct["Academic Period"],
        y = [-10] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        visible = is_visible,
        showlegend = (False),
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    ),
        dict(
            text = "<i>Total Enrollment:</i>",
            font_size = 10,
            showarrow = False,
            xref = "paper",
            x = -.045,
            yref = "y",
            y = -9
        )],
        shapes=[dict(
            type="rect", #making it a rectangle
            xref="paper", #calling to be the shape of the paper
            yref="y", #use the y axis for reference
            x0=-.05, #leftmost side
            x1=1, #rightmost side
            y0=-15, #box bottom
            y1=-3, #box top
            fillcolor="white", #want it to be right
            line=dict(width=1, color = "black"), #I want a thin border around the box
            layer="below", #this will make it be below the gridlines, , but you can still see the numbers
        )
    ],
        )

ratelines.show()

OK - now it is looking nice! I'm wondering about bringing back the legend and maybe tweaking the text. Total Enrollment looks a bit wonky, and having it in the legend may help clear that up. I'm gonna do a version with that first.

In [574]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"],
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"],
        y = [-10] * len(coursepct["Academic Period"]),
        name = "Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )],
        shapes=[dict( #I removed the annotation for enrollment and specified the shape
            type="rect", #making it a rectangle
            xref="paper", #calling to be the shape of the paper
            yref="y", #use the y axis for reference
            x0=0, #leftmost side
            x1=1, #rightmost side
            y0=-15, #box bottom
            y1=-3, #box top
            fillcolor="white", #want it to be right
            line=dict(width=1, color = "black"), #I want a thin border around the box
            layer="below", #this will make it be below the gridlines, , but you can still see the numbers
        )
    ],
        )

ratelines.show()

OK - now that looks like what I was wanting all along! However, I think it clutters the bottom by having two numbers there. I feel like I should make them be the actual terms written out (Spring 2026, Fall 2025, etc.). I'm going to do a little bit of mapping below.

In [575]:
termcodes = mdjmtpct["Academic Period"].unique()
termcodes

[202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610]
Categories (8, int64): [202280, 202310, 202380, 202410, 202480, 202510, 202580, 202610]

In [576]:
termcodesfull = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025", "Spring 2026"]

In [577]:
mapterms = dict(zip(termcodes, termcodesfull))

In [578]:
mdjmtpct["Academic Period"] = mdjmtpct["Academic Period"].map(mapterms)
mdjmtpct.head()

,Course Code,Subject,Academic Period,Subject College,Department,MT C or Higher,"MT C-, D, F, W",MT Not Reported,Total Grades,# of Passing Grades,% of Passing Grades,# of Failing Grades,% of Failing Grades,# of Unreported Grades,% of Unreported Grades
631,MDJ 22796,MDJ,Fall 2022,CCI,MDJ,44.0,10.0,5.0,59.0,0.745763,74.58,0.169492,16.95,0.084746,8.47
214,MDJ 28950,MDJ,Fall 2022,CCI,MDJ,64.0,6.0,8.0,78.0,0.820513,82.05,0.076923,7.69,0.102564,10.26
612,MDJ 28436,MDJ,Fall 2022,CCI,MDJ,36.0,7.0,5.0,48.0,0.750000,75.00,0.145833,14.58,0.104167,10.42
103,MDJ - Term Total,MDJ,Fall 2022,CCI,MDJ,1058.0,171.0,160.0,1389.0,0.761699,76.17,0.123110,12.31,0.115191,11.52
605,MDJ 21449,MDJ,Fall 2022,CCI,MDJ,15.0,4.0,1.0,20.0,0.750000,75.00,0.200000,20.00,0.050000,5.00


In [579]:
#Changing the number of student variable
courses = mdjmtpct["Course Code"].unique() 
courses = sorted(courses, key=lambda x: x != "MDJ - Term Total") #This is the new code that Gemini proposed. This just puts the Term Total to the front. 
# the key = lambda x (based on my understanding) is saying to prioritize the MDJ - Term Total and put that first, and everything after it
coursebuttons = []

ratelines = go.Figure()

for i, course in enumerate(courses): 
    coursepct = mdjmtpct[mdjmtpct["Course Code"] == course] 
    is_visible = (i == 0)

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Passing Grades"],
        name = "MT C or Higher",
        line = dict(color = "green", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C or Higher"], #Calling this out for the custom data piece - it'll specify how many are in this bucket
        hovertemplate = (#I can't comment below due to the slash, but the {} pieces pull the values defined above. 
            #The last bit - <extra></extra> - is something I found searching online that gets rid of the little box next to it that tells you what the value is (https://plotly.com/python/hover-text-and-formatting/)
            "<b>MT C or Higher Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Failing Grades"],
        name = "MT C-, D, F, W",
        line = dict(color = "red", width = 2),
        visible = is_visible,
        customdata = coursepct["MT C-, D, F, W"],
        hovertemplate = (
            "<b>MT C-, D, F, W Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"\
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        x = coursepct["Academic Period"],
        y = coursepct["% of Unreported Grades"],
        name = "MT Not Reported",
        line = dict(color = "#AAAAAA", width = 2), #I slightly tweaked the color here - I realized that it made the text in the hover template harder to read
        visible = is_visible,
        customdata = coursepct["MT Not Reported"],
        hovertemplate = (
            "<b>MT Not Reported Details</b>"\
            "<br>Term: %{x}"\
            "<br>Rate: %{y}%"\
            "<br># of Students: %{customdata} students"
            "<extra></extra>"
        )
    ))

    ratelines.add_trace(go.Scatter(
        mode = "text",
        text = coursepct["Total Grades"],
        textfont = dict(weight = "bold"),
        x = coursepct["Academic Period"],
        y = [-10] * len(coursepct["Academic Period"]),
        name = "Total Student Enrollment (n)",
        visible = is_visible,
        showlegend = (True),
    ))

    visibility = [False] * (len(courses) * 4)
    visibility[i*4: i*4 + 4] = [True, True, True, True]

    buttons = dict(
        label = course,
        method = "update",
        args = [{"visible": visibility}]
    )

    coursebuttons.append(buttons)

    ratelines.update_layout(
            xaxis = dict(
            title = dict(text = "<i>Semester</i>"), #decided to make it italics - had to use HTML code
            type = "category", 
        ),
        yaxis = dict(
            range = [-15,105],
            title = dict(text = "<i>Percentage (%)</i>"), #decided to make it italics - had to use HTML code
        ),
        title=dict(text= "Grade Rates Over Time",
            font_weight = "bold", 
            xanchor = "center",
            x = .5,
            yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
            y = .93),
        updatemenus = [dict(
            type = "dropdown",
            direction = "down",
            showactive = True,
            xanchor = "center",
            yanchor = "top",
            x = .5,
            y = 1.17, #very minor tweak to the y axis
            buttons = coursebuttons)],
        legend = dict(
            title = dict(text = "Legend:", font_weight = "bold"), #decided to add a title to the legend
            xanchor = "center",
            x = .5,
            yanchor = "bottom", 
            y = -.33, #giving a bit of breathing room to the legend
            orientation = "h" 
        ),
        annotations=[dict( #I wanted to add a label for the buttons - had to use ref = paper since we have a figure and align to the whole Canvas
        text = "Course:", 
        font_weight = "bold",
        showarrow = False,
        xref = "paper",
        x = .425,
        yref = "paper",
        y = 1.13
    )],
        shapes=[dict(
            type="rect", #making it a rectangle
            xref="paper", #calling to be the shape of the paper
            yref="y", #use the y axis for reference
            x0=0, #leftmost side
            x1=1, #rightmost side
            y0=-15, #box bottom
            y1=-3, #box top
            fillcolor="white", #want it to be right
            line=dict(width=1, color = "black"), #I want a thin border around the box
            layer="below", #this will make it be below the gridlines, , but you can still see the numbers
        )
    ],
        )

ratelines.show()

There we go! With that, I think I can say that this figure can be put to rest (for now). 

Some things I want to note before I transition to the next figure.
- I am going to want to move the "term translation" up much earlier in my code. I may also want to consider making a for loop for it - that way when new terms are added it isn't a thing to manually do.
- I've thought a lot about this, and I'm thinking when I do my "final" code I want the first section to be my imported libraries and (if possible) a "Change This" variable block. I'm thinking this way all the things that need changed are front and center in the code so you don't have to go seeking all the changes. 
- Once everything is done, I'm going to work on making a "style guide" to make my figures uniform. Thankfully we have a university style guide that I can adhere to, so I may try to stick to that as much as possible.

**Interventions Needed Over Time**

After that last one, this one should be a bit easier. 

We're just looking at the total number of interventions over time. We will need to bring over some of the code from the table project last week so we can plug those numbers into the tables. The nice thing is that we are (once again) making multiple, smaller figures to show the progress over time.

The biggest "issue" with this one will be the level of customization that will be needed when this moves to the version with actual data, as there are more than 2 groups of interventions for some colleges (due to having higher grade requirements to move onto future courses). However, this issue doesn't come up until later, so the goal for now would be to make this in a way that those features could be plugged in later (albeit with some manipulation of location and text).

In [597]:
#Pulling the data in
mtinterfull = pd.read_csv("mthubinterventions.csv")
mtinterfull.head()

,Unnamed: 0,Record ID,Registration Status,Subject,Course,Campus,Final Grade,Mid Term Grade,Department,Major,...,Class,Course Code,Major College,Subject College,Withdrew No MT,Dropped no MT,Dropped no Fin,Final Grade Number,Mid Term Grade Number,MT Grade Status
0,11,11,Registered,AED,22860,KC,NaN,C,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,NaN,2.0,FR/SO Intervention Needed
1,14,14,Registered,AED,22860,KC,C,D,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.0,1.0,FR/SO Intervention Needed
2,19,19,Registered,AED,22860,KC,B-,C,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,2.7,2.0,FR/SO Intervention Needed
3,31,31,Stopped Attending - Failed,AED,22860,KC,W,D,ARCH,ID,...,SO,AED 22860,CAED,CAED,False,False,False,0.0,1.0,FR/SO Intervention Needed
4,33,33,Registered,AED,22860,KC,B-,C-,ARCH,ID,...,FR,AED 22860,CAED,CAED,False,False,False,2.7,1.7,FR/SO Intervention Needed


In [598]:
mtinterfull["MT Grade Status"].value_counts()

MT Grade Status
FR/SO Intervention Needed    5460
JR/SR Intervention Needed     652
Name: count, dtype: int64

In [599]:
intercountpivot = mtinterfull.pivot_table(index=["Academic Period", "Major College"],
                            columns = "MT Grade Status",
                            values = "Record ID",
                            aggfunc = "count").reset_index()

intercountpivot.head()

MT Grade Status,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed
0,Fall 2022,CAED,133,15
1,Fall 2022,CCI,209,49
2,Fall 2022,CotA,424,37
3,Fall 2023,CAED,157,6
4,Fall 2023,CCI,210,26


In [600]:
intercountpivot["Total # of Instances"] = intercountpivot["FR/SO Intervention Needed"] + intercountpivot["JR/SR Intervention Needed"]
intercountpivot.head()

MT Grade Status,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,Fall 2022,CAED,133,15,148
1,Fall 2022,CCI,209,49,258
2,Fall 2022,CotA,424,37,461
3,Fall 2023,CAED,157,6,163
4,Fall 2023,CCI,210,26,236


Below is the biggest adjustment from the original code to now is the removal of the term from the checks. Since we are looking at all the terms, we do not need to specify that any longer.

In [601]:
intercountpivot = intercountpivot.sort_values("Academic Period")
intercountpivot

MT Grade Status,Academic Period,Major College,FR/SO Intervention Needed,JR/SR Intervention Needed,Total # of Instances
0,Fall 2022,CAED,133,15,148
1,Fall 2022,CCI,209,49,258
2,Fall 2022,CotA,424,37,461
3,Fall 2023,CAED,157,6,163
4,Fall 2023,CCI,210,26,236
5,Fall 2023,CotA,442,30,472
6,Fall 2024,CAED,165,11,176
7,Fall 2024,CCI,181,35,216
8,Fall 2024,CotA,406,51,457
10,Fall 2025,CCI,180,22,202


In [602]:
caedintercheck = (intercountpivot["Major College"] == "CAED")
cciintercheck = (intercountpivot["Major College"] == "CCI")
cotaintercheck = (intercountpivot["Major College"] == "CotA")

In [593]:
caedinter = intercountpivot[caedintercheck]
cciinter = intercountpivot[cciintercheck]
cotainter = intercountpivot[cotaintercheck]

OK - so we now have all of the different college intervention tables pulled in. My main concern is with the x axis of the figure. When I set up the previous line chart, the terms were numerical and could be sorted. I could sort the table earlier, but I did some looking around on the plotly site and found a reference on the categorical axes page (https://plotly.com/python/categorical-axes/) that you can call update_xaxes, and then set up the category array in there. Calling back to what I said before about defining all my variables early on - I think this would be a prime candidate for that (can remove old terms and add new ones so the visualization isn't *too* clunky).

Just like before, I'm going to tinker with a single college, knowing that this will be turned into a unique figure for each college in the "final" version of the project.

In [604]:
interlines = go.Figure(data = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"]
))

interlines.show()

So that worked (at least, in terms of ordering the data). However, this does bring up a few things that will need tackled too:
1. I need to set firm tick marks for the side - It starts below 80, so I'm think setting the range to be 0 - 200. I'll need to tinker with the tick range. Torn between 20, 40, and 50 right now, but I'm leaning more toward 40 since that should more accurately catch the numbers.
2. I'll want to add some form of annotation to Spring 2026 to indicate that the intervention criteria shifted in that term (From C- or less to D or less). That'll help explain the large drop in outreach
4. Doing the general "prettying up" of the figure - adding a title, the legend, y and x axis labels, etc.

I'm going to tinker with the sorting first - the rest should be pretty easy to do afterwards!

For sorting the data - it would be nicer if it was numerical. I could sort it first, and then do the mapping after. However, I found something called pd.Categorical that seems to let me do that with categorical data. (https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html). Either way, I'll want to keep this logic in mind when I move to the "final" version of my

In [589]:
#Using categorical to sort the data
caedinter["Academic Period"] = pd.Categorical(caedinter["Academic Period"], #specifying the column we are looking at 
                                              categories = ["Fall 2022", "Spring 2023", "Fall 2023", "Spring 2024", "Fall 2024", "Spring 2025", "Fall 2025", "Spring 2026"], #Listing out the terms - could probably make a variable for this in the future
                                              ordered = True) #Saying that we want to follow the order I set above

caedinter = caedinter.sort_values("Academic Period")#Need to tack this onto the end - this is saying sort the whole table by the values in Academic Period

C:\Users\thilber2\AppData\Local\Temp\ipykernel_47172\3097024691.py:2: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  caedinter["Academic Period"] = pd.Categorical(caedinter["Academic Period"], #specifying the column we are looking at


In [590]:
#Since we ordered them above, this should work now

interlines = go.Figure(data = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"]
))

interlines.show()

There we go! Now it works - we can now safely shift gears and fill it out with the other data and make it look nicer too!

In [ ]:
#Since we're not going through those loops, we can just write the figures like normal (defining traces, then layout, and then plugging it into the figure)

frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2)) #Opted to use institutional colors for these - no strong reason other than theming!

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2))

layout = go.Layout(
    title = "Interventions Needed Over Time",
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(title = "<i># of Intervention</i>"))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.show()

OK - so we've got something that is close to what the "final" version may look like. A couple of things to note for this to improve:
- Due to the data, it adjusted the tick marks for me to be every 50 - minus the top part. I like this a lot, so I think I'm going to adjust it so it starts at 0, and goes up to 200. 
- I need to tweak the title so it becomes centered with the figure
- Putting the legend at the bottom again would be nice - I liked how that looked in my other figure.
- I still need to add that annotation to the Spring 2026 data
- I want to update the hover text for the markers - I think the template that was made for the last figure is applicable here too!

In [ ]:
frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Opted to use institutional colors for these - no strong reason other than theming!
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>"))

layout = go.Layout(
    title = dict(text = "Interventions Needed Over Time",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(
        title = "<i># of Intervention</i>",
        range = [0, 200],
        dtick = 50),
    legend = dict( #Gonna keep this mostly the same - no need to reinvent the wheel!
        title = dict(text = "Legend:", font_weight = "bold"),
        xanchor = "center",
        x = .5,
        yanchor = "bottom", 
        y = -.33,
        orientation = "h" 
        ))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.show()

There we go - that helped make them look a lot nicer. The last thing that I want to add is the annotation piece!

In [ ]:
frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Opted to use institutional colors for these - no strong reason other than theming!
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>"))

layout = go.Layout(
    title = dict(text = "Interventions Needed Over Time",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(
        title = "<i># of Intervention</i>",
        range = [0, 200],
        dtick = 50),
    legend = dict( #Gonna keep this mostly the same - no need to reinvent the wheel!
        title = dict(text = "Legend:", font_weight = "bold"),
        xanchor = "center",
        x = .5,
        yanchor = "bottom", 
        y = -.33,
        orientation = "h" 
        ))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.add_annotation(
    x= caedinter["Academic Period"] == "Spring 2026", #Thinking this should work - finding where it is on the list
    y = 84, #Since the y axis is a count, put the value for the y axis here
    text = "test"
)

interlines.show()

Gotta slide this over a little bit

In [ ]:
frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Opted to use institutional colors for these - no strong reason other than theming!
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<extra></extra>"))

layout = go.Layout(
    title = dict(text = "Interventions Needed Over Time",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(
        title = "<i># of Intervention</i>",
        range = [0, 200],
        dtick = 50),
    legend = dict( #Gonna keep this mostly the same - no need to reinvent the wheel!
        title = dict(text = "Legend:", font_weight = "bold"),
        xanchor = "center",
        x = .5,
        yanchor = "bottom", 
        y = -.33,
        orientation = "h" 
        ))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.add_annotation(
    x= 7, #Realized it may work better to try and reference its index number by hardcoding the 7
    y = 84, #Since the y axis is a count, put the value for the y axis here
    text = "test"
)

interlines.show()

OK - so I have the annotation location right. However, I realized writing out what the criteria shift was may not be helpful. To that end, I'm going to add to the hover template what the criteria was (while also fleshing out the annotation)

In [ ]:
frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Opted to use institutional colors for these - no strong reason other than theming!
    customdata = ["Criteria: MT Grade of D+ or lower" if x == "Spring 2026" else "Criteria: MT Grade of C or below" for x in caedinter["Academic Period"]],
    #Note on the above - wrote it out so we could track when the switch happened. This logic will get more complicated when we add future terms (maybe using or logic?)
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>%{customdata}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>Criteria: MT Grade of F"
    "<extra></extra>"))

layout = go.Layout(
    title = dict(text = "Interventions Needed Over Time",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(
        title = "<i># of Intervention</i>",
        range = [0, 200],
        dtick = 50),
    legend = dict( #Gonna keep this mostly the same - no need to reinvent the wheel!
        title = dict(text = "Legend:", font_weight = "bold"),
        xanchor = "center",
        x = .5,
        yanchor = "bottom", 
        y = -.33,
        orientation = "h" 
        ))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.add_annotation(
    x= 7, #Realized it may work better to try and reference its index number by hardcoding the 7
    y = 84, #Since the y axis is a count, put the value for the y axis here
    text = "Change in"\
         "<br>FR/SO Criteria"
)

interlines.show()

So we are now better showing the criteria for the terms, and highlighting why there is a drop in the Spring 2026 term. However, I want to do some work with the annotation so it looks nicer. I think I have 2 issues with it - the font size and location. I think if I can bump it up some (so it is not overlapping the line), that should take care of my issue. Looking at the plotly documentation, adjusting the ax and ay values do this (https://plotly.com/python/text-and-annotations/#styling-and-coloring-annotations)

In [ ]:
frsotrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["FR/SO Intervention Needed"],
    name = "FR/SO Interventions Needed",
    line = dict(color = "#003976", width = 2), #Opted to use institutional colors for these - no strong reason other than theming!
    customdata = ["Criteria: MT Grade of D+ or lower" if x == "Spring 2026" else "Criteria: MT Grade of C or below" for x in caedinter["Academic Period"]],
    #Note on the above - wrote it out so we could track when the switch happened. This logic will get more complicated when we add future terms (maybe using or logic?)
    hovertemplate = (
    "<b>FR/SO Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>%{customdata}"\
    "<extra></extra>")) 

jrsrtrace = go.Scatter(
    x = caedinter["Academic Period"],
    y = caedinter["JR/SR Intervention Needed"],
    name = "JR/SR Interventions Needed",
    line = dict(color = "#EFAB00", width = 2),
    hovertemplate = (
    "<b>JR/SR Interventions</b>"\
    "<br>Term: %{x}"\
    "<br># of Interventions Needed: %{y}"\
    "<br>Criteria: MT Grade of F"
    "<extra></extra>"))

layout = go.Layout(
    title = dict(text = "Interventions Needed Over Time",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Semester</i>"),
    yaxis = dict(
        title = "<i># of Intervention</i>",
        range = [0, 200],
        dtick = 50),
    legend = dict( #Gonna keep this mostly the same - no need to reinvent the wheel!
        title = dict(text = "Legend:", font_weight = "bold"),
        xanchor = "center",
        x = .5,
        yanchor = "bottom", 
        y = -.33,
        orientation = "h" 
        ))

interlines = go.Figure(data = [frsotrace, jrsrtrace], layout = layout)

interlines.add_annotation(
    x= 7, #Realized it may work better to try and reference its index number by hardcoding the 7
    y = 84, #Since the y axis is a count, put the value for the y axis here
    ax = 0, #I just want it to not overlap, so I figure making this 0 and adjusting the y would make the criteria move up!
    ay = -40, #negative numbers make it go up (not sure why though...)
    font = dict(size = 10),
    text = "Change in"\
         "<br>FR/SO Criteria",
)

interlines.show()

I decided to combo the moving of the text and adjusting the size and wow it looks better. Definitely going to keep it as is.

With that - I think this figure is in a good shape for now! I think adding the criteria to the hovertemplate is going to work wonders for the data. Looking ahead, I'm envisioning this visualization sitting on a page that has the line chart, a (condensed) version of the intervention criteria table, and a button that links out to the contact sheet made in Excel. I think this would provide the necessary information to viewers (showing changes over time, give a quick view of the most recent MT grades, and access to the contact sheet), without overwhelming them.

**Heat Maps**

This is probably going to be the hardest visualization to make, mostly because the APIs make it seem very complicated. It also has become the visualization that I have tackled with the most in terms of "usefulness." The other visualizations I've made have a clear purpose - either showing flucutations in time or a snapshot of a specific term. The purpose of this visualization is to show how a MT grade has increased/decreased based on their final grade. However, it is riddled with limitations inherent to the visualization itself:
- Since we are focusing on interventions, we are considering advisors as the sole reason for the change. However, there are multiple reasons that they could improve a grade (spoke to instructor, got tutoring, started a peer group, was just having a rough time during MTs). Granted, advisors could be repsonsible for connecting them to the resources that the improvement came from. A big asterisk will be needed with this data point.
- I would like to add some kind of box around those who did v. did not get interventions. However, there may be students who are passing one eligible class v. not another.

As such, I think I need to reframe my way of thinking about this visualization. I think I need to view it as **are MT grades an indicator of Final grades** rather than **How effective interventions were in improving grades**. I think the latter is a secondary benefit of this visualization, so maybe enclosing just those grades (i.e. students who qualify for interventions), would be good to meet that secondary purpose, while the whole visual helps with the primary one. 

Through this, I've talked myself into keeping the visualization. The dataset that we will be considering is just the students with majors in the college, and we will have buttons based on *Subject*

In [ ]:
mtheat = pd.read_csv("mthubonlyclean.csv")
mtheat.shape

(58787, 21)

In [ ]:
mtheat.keys() #Running this for a quick reference of the data points

Index(['Unnamed: 0', 'Record ID', 'Registration Status', 'Subject', 'Course',
       'Campus', 'Final Grade', 'Mid Term Grade', 'Department', 'Major',
       'Academic Period', 'Class', 'Course Code', 'Major College',
       'Subject College', 'Withdrew No MT', 'Dropped no MT', 'Dropped no Fin',
       'Final Grade Number', 'Mid Term Grade Number', 'MT Grade Status'],
      dtype='str')

So the main columns we are going to need are Reg Status, Department, Major, Academic Period, Course Code, Subject College, Final Grade Number, Mid Term Grade Number.

Another thing we will want to do with this dataset is remove the Spring 2026 grades. Right now, there are no final grades for the Spring 2026 term, so there are no comparisons that could be made to see how they improved/worsened. In the future, I think I'll make a variable that discounts the most recent term WITHOUT final grades (one variable for MT current term, another for Final Current Term), that way it can be dropped automatically and not impact the dataset.

In [ ]:
mtheat["Academic Period"].unique() #Confirming how the term is written

<StringArray>
[  'Fall 2022', 'Spring 2023',   'Fall 2023', 'Spring 2024',   'Fall 2024',
 'Spring 2025',   'Fall 2025', 'Spring 2026']
Length: 8, dtype: str

In [ ]:
mtheat = mtheat.loc[mtheat["Academic Period"] != "Spring 2026"] #Pulling the rows where Academic Period isn't Spring 2026
mtheat["Academic Period"].unique() #Making sure it was dropped

<StringArray>
[  'Fall 2022', 'Spring 2023',   'Fall 2023', 'Spring 2024',   'Fall 2024',
 'Spring 2025',   'Fall 2025']
Length: 7, dtype: str

In [ ]:
mtheat["Final Grade Number"].unique() #running to have a reference for the numbers

array([3.7, 4. , 0. , 3. , nan, 1.3, 2. , 2.7, 3.3, 1.7, 2.3, 1. ])

Now that those are gone, we can actually work on the heat map. Like normal, we are going to be referencing the plotly documentation (https://plotly.com/python/heatmaps/) for starting with the heat map. I'm going to just plug in some basic values for now to get a feel for how it works.

In [ ]:
testheat = go.Figure(data = go.Heatmap(
    z = [mtheat["Mid Term Grade Number"], mtheat["Final Grade Number"]], #Seems like you plug the values you are comparing here
    x = [0.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.7, 3.0, 3.3, 3.7, 4.0], #you put the labels for X here
    y = [0.0, 1.0, 1.3, 1.7, 2.0, 2.3, 2.7, 3.0, 3.3, 3.7, 4.0] #you put the labels for Y here
))

testheat

Well that's certainly a bunch of colors and lines - but not what we are looking for at all. Reading over the figure, it almost seems like it is trying to read the Z values as the values, with X and Y being the labels for the columns and rows. What's interesting is that it didn't seem to apply the labels down below. I'm wondering if it is due to them being written as numbers and not categories. I'm going to try something real quick.

In [ ]:
testheat = go.Figure(data = go.Heatmap(
    z = [mtheat["Mid Term Grade Number"], mtheat["Final Grade Number"]], #Seems like you plug the values you are comparing here
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], #putting them as text
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"]  #putting them as text
))

testheat

So that worked a bit better! I think before it was trying to find distance between the numbers. However, it still isn't catching and showing concentrations of data (I think). I did some deeper looking and found that it is basically trying to read two rows of data (not the whole list). What I need to do is get it to count the total number of instances where there is crossover (i.e. how many instances of 0.0, 0.0 and 1.0,1.0 for example). I did some looking and found documentation on pd.Crosstabs (https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) - this seems to do that, where it will call two columns and sum how many times that there are instances of crossing over data. I'm going to try it.

The crosstabs function could also be useful in the intervention tables and the like before - but I think it drops other things we'd need (like college and the like), so pivoting them would be better

In [ ]:
ztest = pd.crosstab(mtheat["Mid Term Grade Number"], mtheat["Final Grade Number"]) #so this is specifying to look at those two columns for the crosstabbing
ztest

Final Grade Number,0.0,1.0,1.3,1.7,2.0,2.3,2.7,3.0,3.3,3.7,4.0
Mid Term Grade Number,,,,,,,,,,,
0.0,2043,186,74,115,164,92,98,113,54,38,74
1.0,258,102,64,91,221,82,95,91,31,29,35
1.3,99,36,36,38,75,65,68,48,24,18,17
1.7,120,57,34,80,124,95,126,118,50,51,32
2.0,172,95,48,109,388,186,259,412,119,88,168
2.3,96,42,37,77,139,156,242,314,175,112,92
2.7,121,50,30,67,177,169,356,552,323,290,207
3.0,175,60,31,74,268,219,443,1382,838,791,816
3.3,102,33,20,50,110,122,242,562,790,1054,974


In [ ]:
ztest = pd.crosstab(mtheat["Final Grade Number"], mtheat["Mid Term Grade Number"]) #had to flip it - current setup has it where MT is the Y and Final is the X - want it the opposite way
ztest

Mid Term Grade Number,0.0,1.0,1.3,1.7,2.0,2.3,2.7,3.0,3.3,3.7,4.0
Final Grade Number,,,,,,,,,,,
0.0,2043,258,99,120,172,96,121,175,102,81,247
1.0,186,102,36,57,95,42,50,60,33,35,53
1.3,74,64,36,34,48,37,30,31,20,18,30
1.7,115,91,38,80,109,77,67,74,50,41,74
2.0,164,221,75,124,388,139,177,268,110,75,163
2.3,92,82,65,95,186,156,169,219,122,107,164
2.7,98,95,68,126,259,242,356,443,242,226,306
3.0,113,91,48,118,412,314,552,1382,562,507,910
3.3,54,31,24,50,119,175,323,838,790,661,892


In [ ]:
#Rerunning the Heat Map now that we have the crosstab

testheat = go.Figure(data = go.Heatmap(
    z = ztest, #Seems like you plug the values you are comparing here
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], #you put the labels for X here
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"] #you put the labels for Y here
))

testheat

So that worked! However, we have a new issue - we just made a sea of blue, with a little bit of difference in the 0,0 and 4,4 sectors. We will need to add some distinction to the cells so we can tell them apart. I'm also curious if we can adjust the hover text to be something different (right now it calls the z, x, and y values). I wonder if we could instead make it say something like "X+Y Quadrant, # of Students = Z". 

I want to first figure out the gaps. I did some looking online and found a thread (https://community.plotly.com/t/plotly-heatmap-edge-color-change/88582) that mentions using xgap and ygap to imitate lines between cells. I'm going to see if that can work.

In [ ]:
#Adding lines

testheat = go.Figure(data = go.Heatmap(
    z = ztest,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    xgap = 1, #this adds some space between the cells on the x axis
    ygap = 1 #same as above, but the y axis
))

testheat

OK - so that worked! I entered 1 arbitrarily - mostly to see if it would do anything - and it did! This will be useful to add some distinction between cells. Next I want to try and toy with the hovertemplate to see what I can do with it

In [ ]:
#Adding hovertemplates

testheat = go.Figure(data = go.Heatmap(
    z = ztest,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}" #now that we have a new element, I can add this (instead of custom data)
        "<extra></extra>"
    )
))

testheat

To further differentiate it, I want to see if I can call the z text into the cells. I am going to call texttemplate from the heat map documentation.

In [ ]:
#Seeing if I can add text

testheat = go.Figure(data = go.Heatmap(
    z = ztest,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}" #This was found in the plotly documentation - https://plotly.com/python/annotated-heatmap/
))

testheat.show()

OK - so now we are looking like a real thing! I'm still going to stick to just this instance for tinkering, but I want to add some different elements to the layout while we have a smaller set of data we are working with

In [ ]:
#Adjusting the layout

heattrace = go.Heatmap(
    z = ztest,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis" #I was trying to find a colorblind friendly scale - this came up as one designed specifically for colorblind people, so wanted to give it a try!
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"))

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

So I did some tweaking to add titles and change the color scale. I really like the scale (it aligns with institutional colors too!), but the 14k result in the top left throws it way off. The plotly documentation calls out using zmin/zmax. I think if I call out

In [ ]:
#Adjusting the Z-Scale

heattrace = go.Heatmap(
    z = ztest,
    zmin = 0, #putting it at 0 - the lowest point
    zmax = 2500, #2500 seems to be a good number since it wouldn't put the other numbers at the brightest either
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis" #I was trying to find a colorblind friendly scale - this came up as one designed specifically for colorblind people, so wanted to give it a try!
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"))

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

That looks a lot better! However, the scale is off now. Since the numbers are visible on the figure, I am going to cut the scale from the code.

In [ ]:
#Dropping the scale
heattrace = go.Heatmap(
    z = ztest,
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False #This turns it off
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"))

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

There we go! I really like what it did there - I feel it reduces the clutter a lot. 

Looking at this design, something I'd like to do in the "final" version of the display is have a table to the right indicating what grade(s) correspond with what number. To this end, I think making this visualization narrower by manipulating the height and width of it. I'm going to tinker with some height and width values below to see if I can get one that would look nice

In [ ]:
#Adjusting the height
heattrace = go.Heatmap(
    z = ztest,
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 900,
    width = 650)

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

In [ ]:
#Too Narrow
heattrace = go.Heatmap(
    z = ztest,
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 700,
    width = 400)

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

In [ ]:
#Increasing width and readjusting height - just right
heattrace = go.Heatmap(
    z = ztest,
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700)

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

I like the last one a lot - I think that is the size we are going to go with!

One last thing we need to do before adding filters - consider what the base state of the graph is and what the filters can do with it. There are a few ideas I have for this:
1. Focus on the college level, with filters by department.
2. Focus on the college level, with filters by level (FR/SO and JR/SR) - an added thing we could do here is include some kind of annotation to each table that highlights the eligible students who got the intervention (maybe some kind of box?)
3. Focus on the college level, with filters by year.
4. Focus on the departmental level, with filters by year.
5. Focus on the departmental level, with filters by level - can add the same annotation as option 2.
6. Focus on the departmental level, with filters by course.

In order to consider which option to go for, we need to go back and consider what the purpose we decided on was. Again, this was **are MT grades an indicator of Final Grades** and **How effective interventions were in improving grades**. Reflecting on this, holes start appearring in the options:
- Option 1 has some merit - it provides a college overview, as well as filters into each department. However, it doesn't allow for filtering over terms.
- Options 2 and 5 allows to see how effective interventions were at each level, just different scaled based on the department.
- Options 3 and 4 lets us apply filters by year, but we run into the issue of not seeing how effective interventions were by department.
- Option 6 gets *really* granular - too much so almost. 

I think something else that has been influencing me is the limitations of PowerBI (see *mttablesm3.ipynb* for more detailed discussion of that). The TL;DR version of it is that, based on my work so far, I can only implement one button native to Plotly that works, while the rest have to live outside of the plotly figure. After much thought and deliberation, I think the best path forward is to make a combo of Option 4 and Option 5. I'll create multiple figures (one for each department, segmented even further into figures for each "year"), with filters for the year in the figure. This will be the most complicated one to carry over to PowerBI (and is making me wish I did Dash lol), but it'll be worth it at the end of everything!

With this in mind, there are a few things I'll need to do:
1. Pull all the unique values for the semesters (and get them in order for the dropdown list to look nice!)
2. Make the Crosstab update to reflect the correct term(s)
3. Make a version of the dataframe for each department

In [ ]:
#Sorting the terms
mtheat["Academic Period"] = pd.Categorical(mtheat["Academic Period"],
                                              categories = ["Fall 2025", "Spring 2025", "Fall 2024", "Spring 2024", "Fall 2023", "Spring 2023", "Fall 2022"],
                                              ordered = True)

mtheat = mtheat.sort_values("Academic Period")

In [ ]:
mtheat["Academic Period"].unique()

['Fall 2025', 'Spring 2025', 'Fall 2024', 'Spring 2024', 'Fall 2023', 'Spring 2023', 'Fall 2022']
Categories (7, str): ['Fall 2025' < 'Spring 2025' < 'Fall 2024' < 'Spring 2024' < 'Fall 2023' < 'Spring 2023' < 'Fall 2022']

In [ ]:
heattermlist = mtheat["Academic Period"].unique() #making the list we will call from

In [ ]:
mtheat["Subject"].unique() #making a list for quick reference for the dataframes

<StringArray>
['ARTH',  'FDM',  'MUS', 'ARTS',   'ID', 'THEA',  'ART',  'MDJ', 'ARCH',
  'VCD', 'CMGT', 'EMAT',  'AED', 'COMM',  'DAN', 'ARCS',  'CCI']
Length: 17, dtype: str

In [ ]:
#Making dataframes for each department
aedheat = mtheat[mtheat["Subject"] == "AED"]
archheat = mtheat[mtheat["Subject"] == "ARCH"]
arcsheat = mtheat[mtheat["Subject"] == "ARCS"]
artheat = mtheat[mtheat["Subject"] == "ART"]
arthheat = mtheat[mtheat["Subject"] == "ARTH"]
artsheat = mtheat[mtheat["Subject"] == "ARTS"]
cciheat = mtheat[mtheat["Subject"] == "CCI"]
cmgtheat = mtheat[mtheat["Subject"] == "CMGT"]
commhea = mtheat[mtheat["Subject"] == "COMM"]
danheat = mtheat[mtheat["Subject"] == "DAN"]
ematheat = mtheat[mtheat["Subject"] == "EMAT"]
fdmheat = mtheat[mtheat["Subject"] == "FDM"]
idheat = mtheat[mtheat["Subject"] == "ID"]
mdjheat = mtheat[mtheat["Subject"] == "MDJ"]
musheat = mtheat[mtheat["Subject"] == "MUS"]
theaheat = mtheat[mtheat["Subject"] == "THEA"]
vcdheat = mtheat[mtheat["Subject"] == "VCD"]

In [ ]:
aedheat.shape #making sure it worked

(378, 21)

In [ ]:
#Making the heat map with tables (Picking on MDJ again)
buttons = []
zmatrix = {} #When I ran this the first time, I had this as [] - realized this had to be a dictionary, not a list

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term] #saying that the unique frames should be by term

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"]) #calling crosstab within the loop - this way it creates a unique crosstab for each term

    zmatrix[term] = termcrosstab.values #this is the values that will be plugged into the z spot later

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above (I think - the args are what I'm always shaky on)
    ))

heattrace = go.Heatmap(
    z = zmatrix[term], #Only new piece here - just plugging in the z crosstab
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .93),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.show()

Nice - it worked! However, a few things come to light now:
1. Setting the min/max won't be as helpful. There are changes every semester, so there could be a good v. bad semester
2. I feel like I should set a different color for "0" values - maybe a light gray? It will depend on the color scale I set.
3. I'll need to adjust the location of the button and add my standard label to it (i.e. Semester:)

I also realized that I left out the FR/SO and JR/SR split - in a way, this is a good thing. It made me realize that there aren't actually that many students, so there'd be a huge swath of 0's for some terms. I'm going to stick with this version, and then make a version with the FR/SO and JR/SR splits to see what comes from it (once all the tinkering is done).

In [ ]:
#Moving the button and adding an annotation
buttons = []
zmatrix = {} #

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = 0,
    zmax = 2500,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .25,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

A new issue we have is the zmax and zmin are now worthless - they will just all be blue since there are none close to 2500. I'm going to just min and mix to see if I can get it to work!

In [ ]:
#Adjusting the zmax and zmin
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zmatrix[term].min(), #calling the lowest value
    zmax = zmatrix[term].max(), #calling the highest value
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = "Cividis",
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .25,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

So the adjustment of the color scale kind of worked - I figured pulling the max and min would help with the varying values. However, we are back to the issue of a sea of blue and one island of yellow. I was trying to think about how to force it, and I remembered what we were shown for Seaborn and looking at color scales. I'm going to try that here to manually select colors - this is from plotly's color scale documentation (https://plotly.com/python-api-reference/generated/plotly.graph_objects.layout.html) - we're going to force the 0th value, and the 1th value, but start triggering the yellow-er colors earlier.

In [ ]:
import seaborn as sns

In [ ]:
sns.color_palette("cividis", 12) #thinking if I do a scale of 12, pull some of the values. 

[(0.0, 0.186915, 0.435532),
 (0.148638, 0.239724, 0.430752),
 (0.243485, 0.294274, 0.423014),
 (0.316941, 0.345842, 0.425512),
 (0.387705, 0.400694, 0.437305),
 (0.455072, 0.456718, 0.458976),
 (0.521643, 0.511367, 0.472639),
 (0.597469, 0.570718, 0.465821),
 (0.675981, 0.632468, 0.446736),
 (0.752886, 0.693766, 0.416472),
 (0.836429, 0.761483, 0.368747),
 (0.923279, 0.832822, 0.295244)]

In [ ]:
sns.color_palette("cividis", 10)[0]

(0.003602, 0.195911, 0.441564)

In [ ]:
sns.color_palette("cividis", 10)[3]

(0.37043, 0.38689, 0.433428)

In [ ]:
sns.color_palette("cividis", 10)[6]

(0.616852, 0.585913, 0.462237)

In [ ]:
sns.color_palette("cividis", 10)[9]

(0.905589, 0.818257, 0.312889)

OK - so the colors correspond to the following hex codes:
- #013271
- #5E636E
- #9D9576
- #E7D150

now what we will do is set custom values. 

In [ ]:
civicustom = [
    [0.0,"#013271"], #color starts at 0
    [0.25,"#5E636E"], #color starts at about the 25th percentile
    [0.5,"#9D9576"], #color starts at the 50th percentile
    [1.0,"#E7D150"] #brightest it can be
]

In [ ]:
#Adjusting the zmax and zmin
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] 
    ))

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zmatrix[term].min(),
    zmax = zmatrix[term].max(),
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom, #putting hte custom colors in
    showscale = False
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .25,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

I think I need to set the numbers lower - going to tweak them first

In [ ]:
civicustom = [
    [0.0,"#013271"],
    [0.1,"#5E636E"], #making these two start sooner (hopefully that gets it going!)
    [0.25,"#9D9576"],
    [1.0,"#E7D150"]
]

In [ ]:
#Adjusting the zmax and zmin
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zmatrix[term].min(),
    zmax = zmatrix[term].max(),
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    showscale = False)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .25,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

I like it! However, I do think the colorbar needs to come back - but instead of numbers, maybe I can manually update the text so it instead says (lower, middle, higher). There is documentation here about it - https://plotly.github.io/plotly.py-docs/generated/plotly.graph_objects.Heatmap.html -  that calls out changing the tick values and the like, so maybe this can work?

In [ ]:
#Bringing back the scale
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

    ztop = zmatrix[term].max() #changed these to be up here so they can be called in tickvals later

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [0, ztop/2, ztop], #had to adjust zmax to be a variable to see if this would work - logic is 0, the midpoint of the max, and the max
        ticktext = ["Lower", "Mid", "Higher"], #  the text for the points
    ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .25,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

I like it - I can add a title too, so I'm going to add that to better explain what is being shown.

In [ ]:
#colorbar title
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

    ztop = zmatrix[term].max() #changed these to be up here so they can be called in tickvals later

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [0, ztop/2, ztop], #had to adjust zmax to be a variable to see if this would work - logic is 0, the midpoint of the max, and the max
        ticktext = ["Lower", "Mid", "Higher"], #  the text for the points
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .2125,#had to manipulate this a bit
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

So it added the title, but it is *really* close to the color bar. I'm going to do some tinkering with the values to see what I can do

In [ ]:
#Moving the bar
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

    ztop = zmatrix[term].max() #changed these to be up here so they can be called in tickvals later

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5], #had to adjust zmax to be a variable to see if this would work - logic is 0, the midpoint of the max, and the max
        ticktext = ["Lower", "Mid", "Higher"], #  the text for the points
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .5,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .2125,#had to manipulate this a bit
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

I like that a touch better! It clarifies what the color changes signify, while not being too intrusive (I think). I'm going to do a final bit of tinkering with the look.

In [ ]:
#Fixing buttons
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}] #saying that z should equal whatever is spat out from the crosstab formula above
    ))

    ztop = zmatrix[term].max() #changed these to be up here so they can be called in tickvals later

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5], #had to adjust zmax to be a variable to see if this would work - logic is 0, the midpoint of the max, and the max
        ticktext = ["Lower", "Mid", "Higher"], #  the text for the points
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top", #adjusting the y location of the title - it was a bit tight with the menu
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .625,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3325,#had to manipulate this a bit
    y = 1.06,
    xref = "paper",
    yref = "paper"
)

testheat.show()

The main thing I tweaked was the button location - it was bothering me since it didn't look centered.

The next step is determining the intervention level factor. Like I touched on before - there is a swath of 0's and the numbers on the fringes (namely 0,0 and 4,4) aren't too high. I want to check how many values there are if we look at *just* FR/SO and JR/SR interventions. 

In [ ]:
mdjheat["MT Grade Status"].value_counts()

MT Grade Status
Passing MT Grade             4815
No MT Grade Entered           658
FR/SO Intervention Needed     516
JR/SR Intervention Needed     104
Dropped Course                100
Withdrew From Course           96
Never Attended Course          10
Name: count, dtype: int64

So based on the math - there isn't a large concentration of students who have the FR/SO Interventions, let alone the JR/SR (barely 1.5% of total entries). In fact, there are more No MT Grade Entered entries than them combined. 

With this in mind, I do not think devoting time to creating smaller tables is fitting. Instead, I am going to try and implement the feature I talked about before (highlighting interventions), but over the entire 0.0 - 2.0 range, indicating students in this range likely received an intervention. I think the best way to indicate this is calling the shape feature (like we did for the line chart earlier) to show what is going on.

In [ ]:
#Adding the shape
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .625,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3325,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    x0 = 0,
    x1 = 5,
    y0 = 0,
    y1 = 10
)

testheat.show()

OK - so the first pass made the shape, but not where I want it. I think I'll need to call negative values in order to get the X where I want it, while higher values to get the Y where I want it.

In [ ]:
#Changing the shape
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .625,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3325,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

Now we are getting somewhere! Seems like they operate in .5 intervals, so adjusting them all accordingly got it. What I'll want to do now is a few things:
- I thought about making the border white - I'll need to change the background color of the figure to black so the gridlines are also black.
- I want to move away from a solid color to a dash

In [ ]:
#Making it different colors and dashes
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,
    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .625,
        y = 1.075
    )])

testheat = go.Figure(data = heattrace, layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3325,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "white",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

That worked much better than expected - however, I now am confronted by "how" to indicate this. I thought about an annotation, but I think it would be weird to have it always showing. I then shifted to having a legend, but heat maps don't have those traditionally (they have the colorbar, which we already have). I'm going to try adding a "blank" legend - I may have to change the colors too.

In [ ]:
#Adding a blank legend
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

legendtrace = go.Scatter(
    x = [None], #No data = no lines (hopefully!)
    y = [None],
    mode = "lines",
    line = dict( #just replicating the line style below
        color = "purple",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.075
    )],
    legend = dict(
        orientation = "h", #now going to replicate what I've been doing for the other legends (i.e. putting them at the bottom)
        xanchor = "center",
        x = .6,
        yanchor = "top",
        y = -.1),
    )

testheat = go.Figure(data = [heattrace, legendtrace], layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "purple",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

So I changed the dash color to purple, and I don't think it looks nice. I want it to pop more. I'm going to find some complementary colors by looking online (i.e. what goes well with blue and yellow)

In [ ]:
#Orange
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

legendtrace = go.Scatter(
    x = [None], #No data = no lines (hopefully!)
    y = [None],
    mode = "lines",
    line = dict( #just replicating the line style below
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.075
    )],
    legend = dict(
        orientation = "h", #now going to replicate what I've been doing for the other legends (i.e. putting them at the bottom)
        xanchor = "center",
        x = .6,
        yanchor = "top",
        y = -.1),
    )

testheat = go.Figure(data = [heattrace, legendtrace], layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

In [ ]:
#Cyan
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

legendtrace = go.Scatter(
    x = [None], #No data = no lines (hopefully!)
    y = [None],
    mode = "lines",
    line = dict( #just replicating the line style below
        color = "#00FFFF",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.075
    )],
    legend = dict(
        orientation = "h", #now going to replicate what I've been doing for the other legends (i.e. putting them at the bottom)
        xanchor = "center",
        x = .6,
        yanchor = "top",
        y = -.1),
    )

testheat = go.Figure(data = [heattrace, legendtrace], layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "#00FFFF",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

In [ ]:
#More Purple
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

legendtrace = go.Scatter(
    x = [None], #No data = no lines (hopefully!)
    y = [None],
    mode = "lines",
    line = dict( #just replicating the line style below
        color = "#FF00FF",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.075
    )],
    legend = dict(
        orientation = "h", #now going to replicate what I've been doing for the other legends (i.e. putting them at the bottom)
        xanchor = "center",
        x = .6,
        yanchor = "top",
        y = -.1),
    )

testheat = go.Figure(data = [heattrace, legendtrace], layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF00FF",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

After considering all my options - I think I'm gonna go with the Orange. It signifies a warning, but isn't very distracting either.

In [ ]:
#Orange
buttons = []
zmatrix = {}

for term in heattermlist:
    termframe = mdjheat[mdjheat["Academic Period"] == term]

    termcrosstab = pd.crosstab(termframe["Final Grade Number"], termframe["Mid Term Grade Number"])

    zmatrix[term] = termcrosstab.values

    buttons.append(dict(
        method = "update",
        label = term,
        args = [{"z": [zmatrix[term]]}]
    ))

    ztop = zmatrix[term].max()

    zbot = zmatrix[term].min()

heattrace = go.Heatmap(
    z = zmatrix[term],
    zmin = zbot,
    zmax = ztop,
    x = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"], 
    y = ["0.0", "1.0", "1.3", "1.7", "2.0", "2.3", "2.7", "3.0", "3.3", "3.7", "4.0"],
    xgap = 1,
    ygap = 1,
    hovertemplate = (
        "<b>Sector Details</b>"\
        "<br>Mid Term Grade: %{x}"\
        "<br>Final Grade: %{y}"\
        "<br># of Students: %{z}"
        "<extra></extra>"
    ),
    texttemplate = "%{z}",
    colorscale = civicustom,
    colorbar = dict(
        tickvals = [5, ztop/2, ztop-5],
        ticktext = ["Lower", "Mid", "Higher"],
        title = "<b>Student Density</b>"
        ))

legendtrace = go.Scatter(
    x = [None], #No data = no lines (hopefully!)
    y = [None],
    mode = "lines",
    line = dict( #just replicating the line style below
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    name = "Intervention Eligible",
    showlegend = True,
)

layout = go.Layout(
    title = dict(text = "Grade Movement from MT to Final",
        font_weight = "bold", 
        xanchor = "center",
        x = .5,
        yanchor = "top",
        y = .965),
    plot_bgcolor = "black", #making it black
    xaxis_showgrid = False, #This will hide the lines - there were random white dots throughout the figure that made it look bad
    yaxis_showgrid = False,
    xaxis = dict(title = "<i>Mid Term Grade</i>"),
    yaxis = dict(title = "<i>Final Grade</i>"),
    height = 800,
    width = 700,

    updatemenus = [dict(
        buttons = buttons,
        direction = "down",
        showactive = True,
        xanchor = "center",
        yanchor = "top",
        x = .6,
        y = 1.075
    )],
    legend = dict(
        orientation = "h", #now going to replicate what I've been doing for the other legends (i.e. putting them at the bottom)
        xanchor = "center",
        x = .6,
        yanchor = "top",
        y = -.1),
    )

testheat = go.Figure(data = [heattrace, legendtrace], layout = layout)

testheat.add_annotation(
    text = "<b>Semester:</b>",
    showarrow = False,
    x = .3125,
    y = 1.06,
    xref = "paper",
    yref = "paper"
)
testheat.add_shape(
    type = "rect",
    line = dict(
        color = "#FF8C00",
        width = 2,
        dash = "dash"
    ),
    x0 = -0.5,
    x1 = 4.5,
    y0 = -0.5,
    y1 = 10.5,
    layer = "above"
)

testheat.show()

With all that said - I'm going to put this one to rest too. I think it does what I set out for it to do (i.e. higlights grade changes over time, but giving special attention to the interventions). I'm going to need to make duplicate versions of this in the future (one for each department), but it will be picked up as part of the move to the final project.

**Milestone 3 Reflection**

This milestone was characterized by learning new things. The first new thing was how to translate plotly figures into PowerBI (see the "mttablesm3.ipynb" file for more info on that). There was a lot of trial and error with moving them over and, once I realized the limitations of the translation, I had to rethink how to make my figures. Thankfully I didn't have to do too much with that - what I wanted to filter by was determined early, so I didn't have to rework a lot of things. I am thankful I realized this early though, as it influenced my thinking while working on the line charts and heat map. I had to consider what was the most important filter for each figure, and then how to work within the confines of PowerBI (typically with their buttons) to get this to work properly. This caused quite a few start and stops when considering what to do next, but I am happy with the end product and the direction it is going in.

The next step for me is translating all of this to PowerBI. I have the language downpat (converting to an HTML/JSON object), I just have to think about the best way to plug it in and make the PowerBI logic "work." Additionally, I am going to look at the style of the figures and see if I can settle on a universal style guide. I have something going already using Blue/Yellow/Gray, so I may continue with that theme and spread it throughout the figures. The same is true for using italics for x/y axis titles, bolding the legend, etc. I just have to add that to each figure. I think I'll focus on listing that out separately and then implement it accordingly.

I have also started thinking about what happens post-class with this project. I've been in talks with leadership about this project (mostly inquiries about what I've been doing), and there is interest in seeing this project when finished. I am looking at making a "roadshow" of this dashboard, gathering thoughts on its presentation and what can be improved on/removed to make it the best version of the tool it can be. I have also thought about future-proofing the code so others can run it in case I am absent (which will be the case in Fall 2026). I do not think all of this will be done by the final project (hence the pending name of the final project is *FakeMTCode*), but I at least want to lay the groundwork for this in the future!